# Tái lập 3D-HyperGAMO trên Indian Pines (16 lớp)

Notebook này chạy lại quy trình 3D-HyperGAMO của bài báo trên giao thức DIP
của tác giả. Mục đích là tạo một kết quả nền có thể kiểm tra được trước khi
viết phần cải tiến của khóa luận.

**Giữ nguyên:** kiến trúc Generator/Discriminator/Classifier, cách tạo mẫu,
thứ tự cập nhật, hàm mất mát MSE, kích thước nhiễu 100, Adam (0,0002; beta1
0,5), 8.000 bước, 16 lớp Indian Pines, mặt nạ DIP và cửa sổ 11×11×40.

**Chỉ bổ sung ở mức kỹ thuật:** tương thích TensorFlow/tf-keras hiện nay,
kiểm tra dữ liệu, smoke test, lưu trạng thái và gói kết quả. Các bổ sung này
không thay đổi thuật toán GAN của bài báo.

## Quy trình một lần chạy

`đọc dữ liệu → áp dụng mặt nạ DIP → PCA 40 và chuẩn hóa → lấy ảnh 11×11 →
smoke test → huấn luyện G/D/C 8.000 bước → sinh mẫu → đánh giá 16 lớp →
lưu ZIP kết quả`.

Chạy **Run All** trên Kaggle với GPU. Nếu phiên bị ngắt, giữ nguyên `RESUME_ZIP`
ở ô cấu hình rồi chạy lại để tiếp tục checkpoint; không đổi cấu hình giữa các
lần chạy.


In [ ]:
# 1. CẤU HÌNH DUY NHẤT
from pathlib import Path
import os, sys, json, hashlib, platform, subprocess, importlib.metadata

CFG = {
    'protocol': 'dip_reproduction',  # giao thức chính để tái lập bài báo
    'accepted_class_ids': list(range(1, 17)),
    'patch_size': 11, 'pca_components': 40,
    'pca_solver': 'full',
    'split_seed': 42, 'validation_fraction': 0.20,
    'train_seed': 42, 'batch_size': 100,
    'max_step': 8000,  # 4000 vòng, mỗi vòng C cập nhật hai lần; không phải 8000 epoch
    'eval_every': 500, 'eval_batch_size': 32,
    'max_minutes': 180,  # giới hạn an toàn; có thể tiếp tục bằng checkpoint
    'gpu_index': 0, 'resume_training': True,
}
RUN_TRAINING = True
EVALUATE_TEST = True
INSTALL_COMPATIBILITY_PACKAGES = True
HSI_PATH = None
GT_PATH = None
RESUME_ZIP = None  # chỉ ZIP do CHÍNH notebook này tạo; không dùng ZIP của notebook 8 lớp/cũ
LOCAL_DATA = Path(r'D:\HUIT\Chương trình học\HK1 ( 2026- 2027)\KLTN - GAN\dataset')
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd() / 'hypergamo_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert CFG['accepted_class_ids'] == list(range(1, 17)), 'Notebook chính phải giữ 16 lớp.'
assert CFG['protocol'] in ['dip_reproduction', 'dip_train_only_validation']
assert 0 < CFG['validation_fraction'] < 0.5
assert CFG['max_step'] > 0 and CFG['max_step'] % 2 == 0
assert CFG['eval_every'] > 0 and CFG['eval_every'] % 2 == 0
assert CFG['max_minutes'] > 0
print(json.dumps(CFG, indent=2, ensure_ascii=False))


## Cách chạy trên Kaggle

1. Bật **GPU T4**.
2. Đính kèm `Indian_pines_corrected.mat` và `Indian_pines_gt.mat` làm input.
   Mặt nạ DIP của tác giả đã được lưu trong notebook, không cần tự chia lại.
3. Chạy từ đầu đến cuối bằng **Run All**. Không chạy riêng ô huấn luyện khi
   chưa chạy các ô chuẩn bị dữ liệu và smoke test.
4. Sau khi hoàn tất, tải ZIP được in ở ô cuối. File quan trọng gồm
   `test_metrics.json`, `test_per_class.csv`, `training_history.csv`,
   `generation_diagnostics.csv`, hình mẫu sinh và `config.json`.

Nếu ô kiểm tra báo thiếu GPU, dừng phiên, bật GPU rồi khởi động lại. Không
chuyển sang CPU vì như vậy không còn là lần chạy tái lập cần báo cáo.


## 2. Môi trường và nguồn mã

Dùng TensorFlow 2.20 + tf-keras 2.20 tương thích Keras cũ, **không trộn Keras 3**.
Ô dưới chỉ cài gói nếu phiên đang thiếu phiên bản cần thiết. Không import TensorFlow vào
tiến trình notebook: các bước mô hình chạy trong tiến trình riêng để giải phóng GPU khi xong.
Việc cài đặt cần Internet nếu gói chưa có; dữ liệu DIP và mã mô hình không cần tải ngoài.


In [ ]:
# Không import TensorFlow trong tiến trình notebook.
required = {'tensorflow': '2.20.0', 'tf-keras': '2.20.0'}
missing = []
for package, wanted in required.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    print(package, installed, '→ yêu cầu', wanted)
    if installed != wanted:
        missing.append(f'{package}=={wanted}')
if missing:
    if not INSTALL_COMPATIBILITY_PACKAGES:
        raise RuntimeError('Cần cài gói tương thích: ' + ' '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)
for package, wanted in required.items():
    assert importlib.metadata.version(package) == wanted

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.io import loadmat
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import joblib
from IPython.display import display, FileLink
print('Python:', platform.python_version(), '| NumPy:', np.__version__)


In [ ]:
DATA_SOURCE = '"""Fixed author DIP masks: retain all 16 classes and report, never hide, overlap."""\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.ndimage import maximum_filter\n\nCLASS_NAMES = [\'Alfalfa\', \'Corn-notill\', \'Corn-mintill\', \'Corn\', \'Grass-pasture\',\n    \'Grass-trees\', \'Grass-pasture-mowed\', \'Hay-windrowed\', \'Oats\', \'Soybean-notill\',\n    \'Soybean-mintill\', \'Soybean-clean\', \'Wheat\', \'Woods\',\n    \'Buildings-Grass-Trees-Drives\', \'Stone-Steel-Towers\']\nDIP_MAPPING = np.array([0, 2, 3, 5, 6, 8, 10, 11, 12, 14, 1, 4, 7, 9, 13, 15, 16])\nEXPECTED_TRAIN = np.array([29, 762, 435, 146, 232, 394, 16, 235, 10, 470, 1424, 328, 132, 728, 291, 57])\nEXPECTED_TEST = np.array([25, 675, 404, 99, 274, 354, 2, 250, 10, 503, 1065, 282, 80, 545, 99, 44])\n\n\ndef digest_file(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef dip_split(gt, encoded_mask, cfg):\n    assert encoded_mask.shape == gt.shape == (145, 145)\n    assert np.issubdtype(encoded_mask.dtype, np.integer)\n    assert encoded_mask.min() >= 0 and encoded_mask.max() <= 16\n    author_train = DIP_MAPPING[encoded_mask].astype(\'int16\')\n    test_labels = np.where(author_train == 0, gt, 0).astype(\'int16\')\n    assert np.array_equal([(author_train == c).sum() for c in range(1, 17)], EXPECTED_TRAIN)\n    assert np.array_equal([(test_labels == c).sum() for c in range(1, 17)], EXPECTED_TEST)\n    mismatch = (author_train > 0) & (author_train != gt)\n    label_audit = dict(train_vs_standard_gt_disagreements=int(mismatch.sum()),\n        assigned_on_gt_background=int(((author_train > 0) & (gt == 0)).sum()),\n        assigned_different_nonzero_label=int((mismatch & (gt > 0)).sum()))\n    # Reproduction keeps author train labels, rather than silently changing 161 samples.\n    labels = np.where(author_train > 0, author_train, test_labels)\n    split = np.full(gt.shape, -1, dtype=\'int8\')\n    split[author_train > 0] = 0\n    split[test_labels > 0] = 2\n    if cfg[\'protocol\'] == \'dip_train_only_validation\':\n        rng = np.random.default_rng(cfg[\'split_seed\'])\n        for cid in range(1, 17):\n            indices = np.flatnonzero(author_train.ravel() == cid)\n            n_val = max(2, int(np.floor(len(indices) * cfg[\'validation_fraction\'])))\n            if n_val > len(indices) - 2:\n                raise ValueError(\'Không đủ train/validation cho lớp \' + str(cid))\n            split.ravel()[rng.permutation(indices)[:n_val]] = 1\n    elif cfg[\'protocol\'] != \'dip_reproduction\':\n        raise ValueError(\'Protocol không hợp lệ.\')\n    coords = [np.argwhere(split == i) for i in range(3)]\n    return split, labels, coords, label_audit\n\n\ndef audit_overlap(split, patch_size):\n    centers = [split == i for i in range(3)]\n    footprints = [maximum_filter(m.astype(\'uint8\'), size=patch_size, mode=\'constant\') > 0 for m in centers]\n    report = {}\n    names = [\'train\', \'validation\', \'test\']\n    for a, b in [(0, 1), (0, 2), (1, 2)]:\n        prefix = names[a] + \'_\' + names[b]\n        report[prefix + \'_shared_centers\'] = int((centers[a] & centers[b]).sum())\n        report[prefix + \'_shared_raw_pixels\'] = int((footprints[a] & footprints[b]).sum())\n        # A patch intersects another patch when center distance <= 2 * radius.\n        near_a = maximum_filter(centers[a].astype(\'uint8\'), size=2 * patch_size - 1, mode=\'constant\') > 0\n        report[prefix + \'_overlapping_centers_in_second_set\'] = int((near_a & centers[b]).sum())\n    assert all(v == 0 for k, v in report.items() if k.endswith(\'_shared_centers\'))\n    # Nonzero RAW footprint overlap is expected here; it is recorded, not asserted away.\n    return report\n\n\ndef extract_patches(cube, coords, patch_size):\n    if patch_size <= 0 or patch_size % 2 != 1:\n        raise ValueError(\'patch_size phải lẻ, dương.\')\n    coords = np.asarray(coords)\n    if coords.ndim != 2 or coords.shape[1] != 2 or not np.issubdtype(coords.dtype, np.integer):\n        raise ValueError(\'Tọa độ cần có dạng (N, 2), số nguyên.\')\n    if len(coords) and ((coords < 0).any() or (coords >= np.array(cube.shape[:2])).any()):\n        raise ValueError(\'Tọa độ ngoài ảnh.\')\n    r = patch_size // 2\n    padded = np.pad(cube, ((r, r), (r, r), (0, 0)), mode=\'constant\')\n    out = np.empty((len(coords), patch_size, patch_size, cube.shape[-1], 1), dtype=\'float32\')\n    for i, (row, col) in enumerate(coords):\n        out[i, ..., 0] = padded[row:row+patch_size, col:col+patch_size]\n    if not np.isfinite(out).all():\n        raise ValueError(\'Dữ liệu có NaN/Inf.\')\n    return out\n'
exec(compile(DATA_SOURCE, 'dip_data.py', 'exec'))


## 3. Đọc ảnh, nhãn và kiểm tra đúng bộ dữ liệu

Ảnh 145×145×200; nhãn chuẩn gồm nền 0 và 16 lớp. Có 10.249 tâm có nhãn.
Mặt nạ DIP là cách chia riêng của tác giả, **không được nhầm với nhãn chuẩn**.
Nguồn cố định: [HSI-datasets](https://github.com/mhaut/HSI-datasets/tree/7f3cd0802e491c344aa5986fcb1f26ce85ffbbba/Classification).


In [ ]:
# 4. ĐỌC VÀ KIỂM TRA DỮ LIỆU GỐC
def resolve_input(explicit, filename):
    if explicit:
        p = Path(explicit)
        if not p.is_file(): raise FileNotFoundError(p)
        return p
    if Path('/kaggle/input').is_dir():
        found = sorted(p for p in Path('/kaggle/input').rglob('*.mat') if p.name.lower() == filename.lower())
    else:
        found = [LOCAL_DATA / filename] if (LOCAL_DATA / filename).is_file() else []
    if len(found) != 1:
        raise RuntimeError(f'Tìm được {len(found)} file {filename}: {found}. Điền đường dẫn cụ thể ở ô 1.')
    return found[0]

hsi_path = resolve_input(HSI_PATH, 'Indian_pines_corrected.mat')
gt_path = resolve_input(GT_PATH, 'Indian_pines_gt.mat')
hsi = loadmat(hsi_path)['indian_pines_corrected'].astype(np.float32)
gt = loadmat(gt_path)['indian_pines_gt']
assert hsi.shape == (145, 145, 200) and gt.shape == (145, 145)
assert np.isfinite(hsi).all() and np.issubdtype(gt.dtype, np.integer)
assert np.array_equal(np.unique(gt), np.arange(17)) and int((gt > 0).sum()) == 10249
print('Ảnh:', hsi_path, hsi.shape)
print('Nhãn:', gt_path, '| Có nhãn:', int((gt > 0).sum()))
source_hashes = {'hsi_sha256': digest_file(hsi_path), 'gt_sha256': digest_file(gt_path)}
print('SHA256:', source_hashes)
class_counts = pd.DataFrame({'class_id': np.arange(1, 17), 'class_name': CLASS_NAMES,
                            'total': [int((gt == c).sum()) for c in range(1, 17)]})
display(class_counts)

assert source_hashes['hsi_sha256'] == 'ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939', 'Khác file ảnh đã kiểm chứng; cần kiểm tra trước khi gọi là DIP reproduction.'
assert source_hashes['gt_sha256'] == '65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c', 'Khác file nhãn chuẩn đã đối chiếu nguồn tác giả.'


## 4. Dữ liệu và mức mất cân bằng

Biểu đồ dưới dùng nhãn chuẩn để mô tả dữ liệu; số mẫu **thực sự dùng train** sẽ được
in riêng sau khi áp dụng DIP. Không lấy độ chênh toàn cảnh thay cho độ chênh train.
Ảnh giả màu chỉ để nhìn cấu trúc, không phải ảnh RGB đo trực tiếp.


In [ ]:
FIGURES = {}
def show_save_later(name, fig):
    FIGURES[name] = fig
    plt.show()

def stretch(a):
    lo, hi = np.percentile(a, [2, 98])
    return np.clip((a - lo) / (hi - lo + 1e-8), 0, 1)

pseudo_rgb = np.stack([stretch(hsi[..., b]) for b in [30, 20, 10]], axis=-1)
class_cmap = ListedColormap([(0, 0, 0, 1)] + [plt.get_cmap('tab20')(i) for i in range(16)])
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].imshow(pseudo_rgb); ax[0].set_title('Ảnh giả màu'); ax[0].axis('off')
ax[1].imshow(gt, cmap=class_cmap, vmin=0, vmax=16); ax[1].set_title('Nhãn chuẩn — 16 lớp'); ax[1].axis('off')
ax[2].bar(class_counts.class_id, class_counts.total)
ax[2].set_xticks(range(1, 17)); ax[2].set_xlabel('ID lớp'); ax[2].set_ylabel('Số pixel có nhãn')
ax[2].set_title('Mất cân bằng trên nhãn chuẩn')
plt.tight_layout(); show_save_later('dataset_overview.png', fig)
print('Nhãn chuẩn: ít nhất/nhiều nhất =', class_counts.total.min(), class_counts.total.max(),
      '| Tỷ lệ =', round(class_counts.total.max() / class_counts.total.min(), 2))
fig, ax = plt.subplots(figsize=(11, 4.5))
for cid in range(1, 17):
    ax.plot(hsi[gt == cid].mean(0), label=f'{cid}: {CLASS_NAMES[cid-1]}')
ax.set_xlabel('Chỉ số dải sau loại bỏ dải nhiễu'); ax.set_ylabel('Giá trị dữ liệu gốc')
ax.set_title('Phổ trung bình từng lớp — chỉ mô tả dữ liệu, không dùng chọn siêu tham số')
ax.legend(ncol=2, fontsize=7); ax.grid(alpha=.2)
plt.tight_layout(); show_save_later('mean_spectra.png', fig)


In [ ]:
# Trực quan khối ảnh: hai trục không gian và một trục chỉ số dải phổ.
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
yy, xx = np.mgrid[0:hsi.shape[0]:4, 0:hsi.shape[1]:4]
for band in [0, 30, 60, 90, 120, 150, 199]:
    colors = plt.get_cmap('viridis')(stretch(hsi[..., band])[::4, ::4])
    ax.plot_surface(xx, yy, np.full_like(xx, band), facecolors=colors,
                    rstride=1, cstride=1, shade=False, alpha=.75)
ax.set_xlabel('Cột'); ax.set_ylabel('Hàng'); ax.set_zlabel('Chỉ số dải')
ax.set_title('Một số lát của khối ảnh siêu phổ, không phải địa hình 3D')
ax.view_init(elev=20, azim=-55)
plt.tight_layout(); show_save_later('hsi_cube_3d.png', fig)


## 5. Cách chia DIP và kiểm tra rò rỉ

Giữ ánh xạ nhãn trong `auxil.py` tác giả. Mặt nạ train có 151 vị trí trên nền nhãn chuẩn
và 10 vị trí mang nhãn khác: **161 bất đồng**. Đây là đặc điểm nguồn DIP đã kiểm tra,
không tự sửa hoặc xóa những vị trí đó trong lần tái lập. Vì vậy tổng train + test là
10.400, không bằng 10.249 nhãn chuẩn. Cần ghi rõ khi báo cáo.

Hai tâm khác nhau chưa đủ đảm bảo hai vùng 11×11 không dùng chung pixel. Audit tính
cả tâm trùng, pixel đầu vào dùng chung và số tâm ở tập sau có vùng ảnh giao tập trước.
Nhánh train-only + validation vẫn dùng cách chia DIP; validation được phân tầng trong
train theo nhãn, **không phải validation tách vùng không gian**.


In [ ]:
DIP_B64 = 'eJzt201vmzAYwHHDWvKyjeS6G7tvUxupk5JbutOkRZq23KtIoRpTl0RNJvVr7BuPBAjEL/AYG9sk/kuozQvwkwvFNOpsOv82vQ/uPt0Es+n842P0FAbfnxa7x/Xzn0mwWW+jlw/Bl+dwsQuXwXo1Cea//gY/w00w+hzcjiej8eRuFIxubseoRs7XmR9/efmBkBd/7cZLL33pOn38L12ceHkXL9FqGS1Wm2gVbh+W0fb3OlrtHpbbcJeu58bL+3h77r46JqxXSeIbciWJJIPERbJBLNHrNOzpbtzJBuSDGCKNoEzkHMJBmEgfKBedoAwB5apLAhFhIEJ08SAHB+Ei/SAEAiH5INcwUHqeUUCUFIBcs0CF30RGgFw9ID/fAo2TPVsEsSYCUkA+A4RfPEgQKZIB8hkgVxPIZ4BcAoRIECEqgLI4QX4joLf7aoF8HhAiQbiIAuLKZ4O05MNAfAfl1Wn0N3VoiYAcJ7+UXJUmE0ScMHRQuUgeiHIGawF5SYkIBioVoWRXhoGKOxUDecfKQKVR910LhCyIDUqeAB9DqkCHMz/JOFAXsykEFeMAkRvPpis8oNMYswmqiBwhJqh+PNOPFoAov5EtCNuhetBwaBJouK8UhBSChmmmgIYwEDJthHBQgwFBlMsE8wVK2R9IIbddZoIKj1lXe6Wg4o2RdhDaD0/hcs8CUachYBBvrQTRpiFtA3E7+EC0O1W5DFGQ9HHhBQFn9QpB6mLdNWrLgqo6dxDwz5EAkCSRBSkCycHUBg0GBoEGh4wBDbLkCcRqF8irCh1nIYyJAf8RLQbyeEEcI0QTQUCIA1TNEQblH0UAQBBPOQggqwDxNwCJAENkQVUiJaDy47lxECkCgTwZoF6cGaBeHgDEUy1Qr9cukJDIgrSAREQtApVMQSxIGCRDZAyoThSQjPp9paDSN/eTzAEh40BIH8ihlYo0gKic5Bw3DmTUWSaYJBA5RhZ0yaDqz9wUH9RQEHnvfNqbOP7PD2GgDmWltoJ4RQKgqowDofR/XxoBdciVwCA+0TmBKn+yjYI6xEpwEJeoWVCNKCDgShJ2fr6gJqsLuqanDcTwaAOxPPJAfCsxPQ2DWGPH9kgDlWLxd5R4FIIK72kWVI0lx0gv6PidASCbzWaz2Ww2pf0H3l+xCA=='
import base64, zlib, io
dip_bytes = zlib.decompress(base64.b64decode(DIP_B64))
assert hashlib.sha256(dip_bytes).hexdigest() == '430c37ce56a160d410378aa136e5e696e2436e2fe03e275f306c3502f622e77b'
encoded_dip = loadmat(io.BytesIO(dip_bytes))['indianpines_disjoint_dset']
split, labels_protocol, coords_by_split, label_audit = dip_split(gt, encoded_dip, CFG)
overlap_report = audit_overlap(split, CFG['patch_size'])
print('Bất đồng nhãn:', label_audit)
print('Chồng lấn:', json.dumps(overlap_report, indent=2))
count_table = class_counts.copy()
for k, name in enumerate(['train', 'validation', 'test']):
    count_table[name] = [int(((split == k) & (labels_protocol == c)).sum()) for c in range(1, 17)]
assert (count_table.train > 0).all() and (count_table.test > 0).all()
count_table['class_deficit'] = count_table.train.max() - count_table.train
display(count_table)
print('Tổng train/val/test:', [len(c) for c in coords_by_split])
print('Tỷ lệ mất cân bằng TRAIN:', round(count_table.train.max()/count_table.train.min(), 2))
print('CẢNH BÁO: DIP trong notebook này có chồng lấn vùng ảnh; không gọi là leakage-free.')
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(split + 1, cmap=ListedColormap(['black', '#2ca02c', '#ffbf00', '#1f77b4']), vmin=0, vmax=3)
ax[0].set_title('Xanh lá: train; vàng: val; xanh dương: test'); ax[0].axis('off')
ax[1].bar(count_table.class_id, count_table.train, label='Train thật')
ax[1].bar(count_table.class_id, count_table.class_deficit, bottom=count_table.train,
          alpha=.4, label='Thiếu hụt so với lớp lớn nhất')
ax[1].set_xticks(range(1, 17)); ax[1].legend(); ax[1].set_title('Thiếu hụt quyết định xác suất sinh theo lớp')
plt.tight_layout(); show_save_later('dip_split_and_imbalance.png', fig)


## 6. Mô hình gốc và cách huấn luyện

Nguồn: [3D-HyperGAMO của Roy và cộng sự](https://github.com/mhaut/3D-HyperGAMO/tree/3f92889aec4a7e2c8f7a2db4c8a9bf167f296f5e),
DOI `10.1109/TGRS.2021.3052048`. Mã gốc được nhúng, chỉ đổi đường import để tương thích.

- **G:** nhận nhiễu 100 chiều và nhãn, qua đầu sinh riêng từng lớp để tạo tổ hợp lồi của
  các vùng ảnh train cùng lớp; không phải GAN ảnh RGB thông thường.
- **C:** bộ phân lớp phổ–không gian với Conv3D và Conv2D.
- **D:** bộ phân biệt có điều kiện theo nhãn.
- Giữ MSE, mục tiêu đối kháng `1 - one_hot` cho G–C, **một Adam dùng chung**
  (lr 0,0002; beta1 0,5), cập nhật C/D trên thật và giả, G học đối kháng với C rồi D.
- Chọn lớp cần sinh theo phần thiếu hụt so với lớp lớn nhất. Sinh trong quá trình học,
  **không phải tạo sẵn một dataset cân bằng đúng bằng nhau**.

Không thêm loss mới, bộ lọc chất lượng hay mô hình thay thế. Lưu `main.py`, `auxil.py`,
`dense_suppli.py` để có thể đối chiếu; bộ chạy riêng chỉ thêm tương thích, ghi log và lưu/nạp.


In [ ]:
AUTHOR_NETWORKS = r'''
# For running in python 2.x
from __future__ import print_function, unicode_literals
from __future__ import absolute_import, division
import numpy as np
import tensorflow as tf
from keras import backend as K
from keras.layers import Input, Dropout, Dense, RepeatVector, Lambda, Reshape, Conv3D, Conv2D, Flatten, InputSpec
from keras.layers import BatchNormalization, Concatenate, Multiply, Add, Conv2DTranspose, GlobalAveragePooling2D
from keras.layers.advanced_activations import LeakyReLU, Softmax
from keras.models import Model

def denseGamoGenCreate(latDim, num_class):
    noise = Input(shape=(latDim, ))
    labels = Input(shape=(num_class, ))
    gamoGenInput = Concatenate()([noise, labels])

    x = Dense(7 * 7 * 256, use_bias=False)(gamoGenInput)
    x = BatchNormalization(momentum=0.9)(x)
    x = LeakyReLU()(x)

    x = Reshape((7, 7, 256))(x)

    x = Conv2DTranspose(
        128, (5, 5), strides=(1, 1), padding='same', use_bias=False)(x)
    x = BatchNormalization(momentum=0.9)(x)
    x = LeakyReLU()(x)

    x = Conv2DTranspose(
        64, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = BatchNormalization(momentum=0.9)(x)
    x = LeakyReLU()(x)

    x = Conv2DTranspose(
        1, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = BatchNormalization(momentum=0.9)(x)

    gamoGenFinal = Flatten()(x)

    gamoGen = Model([noise, labels], gamoGenFinal)
    return gamoGen



def denseGenProcessCreate(numMinor, dataMinor,sh,mul):
    ip1=Input(shape=(784,))
    x=Dense(numMinor, activation='softmax')(ip1)
    x=RepeatVector(mul)(x)
    z = np.reshape(dataMinor,(numMinor,mul))
    genProcessFinal=Lambda(lambda x: K.sum(x*K.transpose(K.constant(z)), axis=2))(x)
    genProcessReshape = Reshape(sh)(genProcessFinal)
    genProcess=Model(ip1, genProcessReshape)
    return genProcess

def denseDisCreate(sh, num_class):
    imIn=Input(shape=sh)
    conv_layer1 = Conv3D(filters=8, kernel_size=(1, 1, 7), activation='relu', padding='same')(imIn)
    conv_layer2 = Conv3D(filters=16, kernel_size=(3, 3, 5), activation='relu',padding='same')(conv_layer1)
    conv_layer3 = Conv3D(filters=32, kernel_size=(5, 5, 7), activation='relu',padding='same')(conv_layer2)
    conv3d_shape = conv_layer3.shape
    conv_layer3 = Reshape((conv3d_shape[1], conv3d_shape[2], conv3d_shape[3]*conv3d_shape[4]))(conv_layer3)
    conv_layer4 = Conv2D(filters=64, kernel_size=(3,3), activation='relu',padding='same')(conv_layer3)
    conv_layer4 = GlobalAveragePooling2D()(conv_layer4)
    flatten_layer = Flatten()(conv_layer4)
    labels=Input(shape=(num_class,))
    disInput=Concatenate()([flatten_layer, labels])
    x=Dropout(0.5)(disInput)
    disFinal=Dense(1, activation='sigmoid', kernel_initializer="he_normal")(x)
    dis=Model([imIn, labels], disFinal)
    return dis

def denseMlpCreate(sh, num_class):
    imIn=Input(shape=sh)
    conv_layer1 = Conv3D(filters=8, kernel_size=(1, 1, 7), activation='relu', padding='same')(imIn)
    conv_layer2 = Conv3D(filters=16, kernel_size=(3, 3, 5), activation='relu',padding='same')(conv_layer1)
    conv_layer3 = Conv3D(filters=32, kernel_size=(5, 5, 7), activation='relu',padding='same')(conv_layer2)
    conv3d_shape = conv_layer3.shape
    conv_layer3 = Reshape((conv3d_shape[1], conv3d_shape[2], conv3d_shape[3]*conv3d_shape[4]))(conv_layer3)
    conv_layer4 = Conv2D(filters=64, kernel_size=(3,3), activation='relu',padding='same')(conv_layer3)
    conv_layer4 = GlobalAveragePooling2D()(conv_layer4)
    flatten_layer = Flatten()(conv_layer4)
    x=Dropout(0.5)(flatten_layer)
    mlpFinal = Dense(num_class, activation="softmax", kernel_initializer="he_normal")(x)
    mlp=Model(imIn, mlpFinal)
    return mlp

'''
print('Đã nạp mã gốc: dense_net3D_SA.py')


In [ ]:
AUTHOR_MAIN = r'''
import os
import auxil
import argparse
import numpy as np
import dense_suppli as spp
import dense_net3D_SA as nt
from keras.optimizers import Adam
from keras.utils.np_utils import to_categorical
from keras.layers import Input
from keras.models import Model



def load_hyper(args):
    if args.dataset not in ["UH", "DIP", "DUP"]:
        acrstate = None if args.random_state == None else args.random_state+args.idtest
        pixelsO, labelsO, numclass = auxil.loadData(args.dataset, num_components=args.components, rand_state=acrstate)
        pixelsO, labelsO = auxil.createImageCubes(pixelsO, labelsO, windowSize=args.spatialsize, removeZeroLabels = False)
        pixels = pixelsO[labelsO!=0]
        labels = labelsO[labelsO!=0] - 1
        bands = pixels.shape[-1]; numberofclass = len(np.unique(labels))
        pixels = pixels.reshape((pixels.shape[0], args.spatialsize, args.spatialsize, bands, 1))
        pixelsO = pixelsO.reshape((pixelsO.shape[0], args.spatialsize, args.spatialsize, bands, 1))
        x_train, x_test, y_train, y_test = auxil.split_data(pixels, labels, args.tr_percent, rand_state=acrstate)
        del pixels, labels
    else:
        x_train, x_test, y_train, y_test, pixelsO, labelsO, bands, numberofclass = auxil.loadDataFIX(args)
    return (x_train, y_train), (x_test, y_test), (pixelsO, labelsO), numberofclass, bands





def main():
    parser = argparse.ArgumentParser(description='PyTorch DCNNs Training')
    parser.add_argument('--max_step', default=8000, type=int, help='number of total epochs to run')
    parser.add_argument('--idtest', default=0, type=int, help='id of experiment')
    parser.add_argument('--lr', '--learning-rate', default=0.1, type=float, help='initial learning rate')
    parser.add_argument('--components', default=None, type=int, help='dimensionality reduction')
    parser.add_argument('--dataset', default='IP', type=str, help='dataset (options: IP, UP, SV, KSC)')
    parser.add_argument('--tr_percent', default=0.10, type=float, help='samples of train set')
    parser.add_argument('--tr_bsize', default=100, type=int, help='mini-batch train size (default: 100)')
    parser.add_argument('--inplanes', dest='inplanes', default=16, type=int, help='bands before blocks')
    parser.add_argument('--no-bottleneck', dest='bottleneck', action='store_false', help='to use basicblock (default: bottleneck)')
    parser.add_argument('--spatialsize', dest='spatialsize', default=7, type=int, help='spatial-spectral patch dimension')
    parser.add_argument('--momentum', default=0.9, type=float, help='momentum')
    parser.add_argument('--weight-decay', '--wd', default=1e-4, type=float, help='weight decay (default: 1e-4)')
    parser.add_argument('--random_state', default=None, type=int, help='random seed')

    args = parser.parse_args()
    state = {k: v for k, v in args._get_kwargs()}

    #args.components = 35 if args.dataset == 'IP' else 15
    (trainS, labelTr), (testS, labelTs), (pixels, labels), numberofclass, bands = load_hyper(args)


    n, m = trainS.shape[0], testS.shape[0]
    labelTr, labelTs, c, pInClass, _, newOrder = spp.relabel(labelTr, labelTs)
    imbalancedCls, toBalance, imbClsNum, ir=spp.irFind(pInClass, c)

    labelsCat = to_categorical(labelTr)

    shuffleIndex=np.random.choice(np.arange(n), size=(n,), replace=False)
    trainS=trainS[shuffleIndex]
    labelTr=labelTr[shuffleIndex]
    labelsCat=labelsCat[shuffleIndex]
    classMap=list()
    for i in range(c):
        classMap.append(np.where(labelTr==i)[0])


    adamOpt=Adam(0.0002, 0.5)
    latDim, modelSamplePd, resSamplePd=100, 8000, 500

    # model initialization
    mlp=nt.denseMlpCreate(sh=(args.spatialsize, args.spatialsize, bands, 1, ), num_class=numberofclass)
    mlp.compile(loss='mean_squared_error', optimizer=adamOpt)
    mlp.trainable=False

    dis=nt.denseDisCreate((args.spatialsize, args.spatialsize, bands, 1, ), num_class=numberofclass)
    dis.compile(loss='mean_squared_error', optimizer=adamOpt)
    dis.trainable=False

    gen=nt.denseGamoGenCreate(latDim, numberofclass)


    gen_processed, genP_mlp, genP_dis=list(), list(), list()
    for i in range(imbClsNum):
        dataMinor=trainS[classMap[i], :]
        numMinor=dataMinor.shape[0]
        gen_processed.append(nt.denseGenProcessCreate(numMinor, dataMinor,sh = \
            (args.spatialsize, args.spatialsize, bands, 1),mul = args.spatialsize * args.spatialsize * bands))

        ip1=Input(shape=(latDim,))
        ip2=Input(shape=(c,))
        op1=gen([ip1, ip2])
        op2=gen_processed[i](op1)
        op3=mlp(op2)
        genP_mlp.append(Model(inputs=[ip1, ip2], outputs=op3))
        genP_mlp[i].compile(loss='mean_squared_error', optimizer=adamOpt)

        ip1=Input(shape=(latDim,))
        ip2=Input(shape=(c,))
        ip3=Input(shape=(c,))
        op1=gen([ip1, ip2])
        op2=gen_processed[i](op1)
        op3=dis([op2, ip3])
        genP_dis.append(Model(inputs=[ip1, ip2, ip3], outputs=op3))
        genP_dis[i].compile(loss='mean_squared_error', optimizer=adamOpt)


    batchDiv, numBatches, bSStore = spp.batchDivision(n, args.tr_bsize)
    genClassPoints=int(np.ceil(args.tr_bsize / c))

    fileStart = './SavedModel/'
    savePath = './SavedModel/'
    fileEnd = '_Model.h5'
    if not os.path.exists(fileStart):
        os.makedirs(fileStart)
    picPath=savePath+'Pictures'
    if not os.path.exists(picPath):
        os.makedirs(picPath)


    iter=np.int(np.ceil(args.max_step/resSamplePd)+1)
    acsaSaveTr, gmSaveTr, accSaveTr=np.zeros((iter,)), np.zeros((iter,)), np.zeros((iter,))
    acsaSaveTs, gmSaveTs, accSaveTs=np.zeros((iter,)), np.zeros((iter,)), np.zeros((iter,))
    confMatSaveTr, confMatSaveTs=np.zeros((iter, c, c)), np.zeros((iter, c, c))
    tprSaveTr, tprSaveTs=np.zeros((iter, c)), np.zeros((iter, c))



    step=0
    bestacc = -1
    while step < args.max_step:
        for j in range(numBatches):
            x1, x2=batchDiv[j, 0], batchDiv[j+1, 0]
            validR=np.ones((bSStore[j, 0],1))-np.random.uniform(0,0.1, size=(bSStore[j, 0], 1))
            mlp.train_on_batch(trainS[x1:x2], labelsCat[x1:x2])
            dis.train_on_batch([trainS[x1:x2], labelsCat[x1:x2]], validR)

            invalid=np.zeros((bSStore[j, 0], 1))+np.random.uniform(0, 0.1, size=(bSStore[j, 0], 1))
            randNoise=np.random.normal(0, 1, (bSStore[j, 0], latDim))
            fakeLabel=spp.randomLabelGen(toBalance, bSStore[j, 0], c)
            rLPerClass=spp.rearrange(fakeLabel, imbClsNum)
            fakePoints=np.zeros((bSStore[j, 0],args.spatialsize, args.spatialsize, bands, 1))
            genFinal=gen.predict([randNoise, fakeLabel])
            for i1 in range(imbClsNum):
                if rLPerClass[i1].shape[0]!=0:
                    temp=genFinal[rLPerClass[i1]]
                    fakePoints[rLPerClass[i1]]=gen_processed[i1].predict(temp)

            mlp.train_on_batch(fakePoints, fakeLabel)
            dis.train_on_batch([fakePoints, fakeLabel], invalid)

            for i1 in range(imbClsNum):
                validA=np.ones((genClassPoints, 1))
                randomLabel=np.zeros((genClassPoints, c))
                randomLabel[:, i1]=1
                randNoise=np.random.normal(0, 1, (genClassPoints, latDim))
                oppositeLabel=np.ones((genClassPoints, c))-randomLabel
                genP_mlp[i1].train_on_batch([randNoise, randomLabel], oppositeLabel)
                genP_dis[i1].train_on_batch([randNoise, randomLabel, randomLabel], validA)

            if step%resSamplePd==0:
                saveStep=int(step//resSamplePd)

                pLabel=np.argmax(mlp.predict(trainS), axis=1)
                acsa, gm, tpr, confMat, acc=spp.indices(pLabel, labelTr)
                print('Train: Step: ', step, 'ACSA: ', np.round(acsa, 4), 'GM: ', np.round(gm, 4))
                print('TPR: ', np.round(tpr, 2))
                acsaSaveTr[saveStep], gmSaveTr[saveStep], accSaveTr[saveStep]=acsa, gm, acc
                confMatSaveTr[saveStep]=confMat
                tprSaveTr[saveStep]=tpr

                pLabel=np.argmax(mlp.predict(testS), axis=1)
                acsa, gm, tpr, confMat, acc=spp.indices(pLabel, labelTs)
                print('Test: Step: ', step, 'ACSA: ', np.round(acsa, 4), 'GM: ', np.round(gm, 4))
                print('TPR: ', np.round(tpr, 2))
                acsaSaveTs[saveStep], gmSaveTs[saveStep], accSaveTs[saveStep]=acsa, gm, acc
                confMatSaveTs[saveStep]=confMat
                tprSaveTs[saveStep]=tpr

                results = auxil.reports(pLabel, labelTs)[2]
                if bestacc <= results[0]:
                    bestacc = auxil.reports(pLabel, labelTs)[2][0]
                    resultsF = auxil.reports(pLabel, labelTs)[2]

            if step%modelSamplePd==0 and step!=0:
                direcPath=savePath+'gamo_models_'+str(step)
                if not os.path.exists(direcPath):
                    os.makedirs(direcPath)
                gen.save(direcPath+'/GEN_'+str(step)+fileEnd)
                mlp.save(direcPath+'/MLP_'+str(step)+fileEnd)
                dis.save(direcPath+'/DIS_'+str(step)+fileEnd)
                for i in range(imbClsNum):
                    gen_processed[i].save(direcPath+'/GenP_'+str(i)+'_'+str(step)+fileEnd)

            step=step+2
            if step>=args.max_step: break

    print(newOrder)
    print(resultsF)

if __name__ == '__main__':
	main()

'''
print('Đã nạp mã gốc: main.py')


In [ ]:
AUTHOR_REFERENCE_FILES = {'auxil.py': 'import os\nimport numpy as np\nimport scipy.io as sio\nfrom sklearn.decomposition import PCA\nfrom sklearn.preprocessing import MinMaxScaler, StandardScaler\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score\nfrom operator import truediv\nimport copy\n\ndef random_unison(a,b, rstate=None):\n    assert len(a) == len(b)\n    p = np.random.RandomState(seed=rstate).permutation(len(a))\n    return a[p], b[p]\n\n\ndef loadDataFIX(args):\n    data, labels, numclass = loadData(args.dataset, num_components=args.components)\n    num_bands = data.shape[-1]\n    pixels, labels = createImageCubes(data, labels, windowSize=args.spatialsize, removeZeroLabels = False)\n    num_class = len(np.unique(labels)) - 1\n    if args.dataset == "UH":\n        data_path = os.path.join(os.getcwd(),\'HSI-datasets/Classification/\')\n        y_train = sio.loadmat(os.path.join(data_path, \'houston_gt.mat\'))[\'houston_gt_tr\'].reshape(-1)\n        y_test = sio.loadmat(os.path.join(data_path, \'houston_gt.mat\'))[\'houston_gt_te\'].reshape(-1)\n        Xtrain = pixels[y_train!=0,:,:,:]\n        Xtest  = pixels[y_test!=0,:,:,:]\n        del pixels\n        y_train = y_train[y_train!=0] - 1\n        y_test  = y_test[y_test!=0] - 1\n        return Xtrain, Xtest, y_train, y_test, num_bands, num_class\n    elif args.dataset == "DIP":\n        y_train2 = sio.loadmat(os.path.join(\'HSI-datasets/Classification/\', \'indianpines_disjoint_dset.mat\'))\\\n                                            [\'indianpines_disjoint_dset\']\n        y_test = sio.loadmat(os.path.join(\'HSI-datasets/Classification/\', \'indian_pines_corrected_gt.mat\'))[\'indian_pines_gt\']\n        y_train = copy.deepcopy(y_train2)\n        for i, val in enumerate([0,2,3,5,6,8,10,11,12,14,1,4,7,9,13,15,16]): y_train[y_train2==i] = val\n        del y_train2\n        y_test[y_train!=0] = 0\n    elif args.dataset == "DUP":\n        y_train = sio.loadmat(os.path.join(\'HSI-datasets/Classification/\', \'TRpavia_fixed.mat\'))[\'TRpavia_fixed\'].reshape(-1)\n        y_test = sio.loadmat(os.path.join(\'HSI-datasets/Classification/\', \'TSpavia_fixed.mat\'))[\'TSpavia_fixed\'].reshape(-1)\n    y_train = y_train.reshape(-1)\n    y_test = y_test.reshape(-1)\n    Xtrain = pixels[y_train!=0,:]\n    Xtest  = pixels[y_test!=0,:]\n    #del pixels\n    y_train = y_train[y_train!=0] - 1\n    y_test  = y_test[y_test!=0] - 1\n    Xtrain, y_train = random_unison(Xtrain,y_train, rstate=None)\n    return Xtrain, Xtest, y_train, y_test, pixels, labels, num_bands, num_class\n\n\ndef split_data(pixels, labels, percent, splitdset="sklearn", with_val=False, rand_state=None):\n    return train_test_split(pixels, labels, test_size=(1-percent), stratify=labels, random_state=rand_state)\n\ndef loadData(name, num_components=None, rand_state=None):\n    data_path = os.path.join(os.getcwd(),\'HSI-datasets/Classification/\')\n    if name in [\'IP\', \'DIP\']:\n        data = sio.loadmat(os.path.join(data_path, \'indian_pines_corrected.mat\'))[\'indian_pines_corrected\']\n        labels = sio.loadmat(os.path.join(data_path, \'indian_pines_corrected_gt.mat\'))[\'indian_pines_gt\']\n    elif name == \'SV\':\n        data = sio.loadmat(os.path.join(data_path, \'salinas_corrected.mat\'))[\'salinas_corrected\']\n        labels = sio.loadmat(os.path.join(data_path, \'salinas_gt.mat\'))[\'salinas_gt\']\n    elif name in [\'UP\', \'DUP\']:\n        data = sio.loadmat(os.path.join(data_path, \'paviaU.mat\'))[\'paviaU\']\n        labels = sio.loadmat(os.path.join(data_path, \'paviaU_gt.mat\'))[\'paviaU_gt\']\n    elif name == \'KSC\':\n        data = sio.loadmat(os.path.join(data_path, \'KSC.mat\'))[\'KSC\']\n        labels = sio.loadmat(os.path.join(data_path, \'KSC_gt.mat\'))[\'KSC_gt\']\n    elif name == \'UH\':\n        data = sio.loadmat(os.path.join(data_path, \'houston.mat\'))[\'houston\']\n        labels = sio.loadmat(os.path.join(data_path, \'houston_gt.mat\'))[\'houston_gt_tr\']\n        labels += sio.loadmat(os.path.join(data_path, \'houston_gt.mat\'))[\'houston_gt_te\']\n    elif name == \'BW\':\n        data = sio.loadmat(os.path.join(data_path, \'Botswana.mat\'))[\'Botswana\']\n        labels = sio.loadmat(os.path.join(data_path, \'Botswana_gt.mat\'))[\'Botswana_gt\']\n    else:\n        print("NO DATASET")\n        exit()\n\n    shapeor = data.shape\n    data = data.reshape(-1, data.shape[-1])\n    if num_components != None:\n        data = PCA(n_components=num_components, random_state=rand_state).fit_transform(data)\n        shapeor = np.array(shapeor)\n        shapeor[-1] = num_components\n    data = MinMaxScaler(feature_range=(-1, 1)).fit_transform(data)\n    data = data.reshape(shapeor)\n    num_class = len(np.unique(labels)) - 1\n    return data, labels, num_class\n\n\ndef padWithZeros(X, margin=2):\n    newX = np.zeros((X.shape[0] + 2 * margin, X.shape[1] + 2* margin, X.shape[2]))\n    x_offset = margin\n    y_offset = margin\n    newX[x_offset:X.shape[0] + x_offset, y_offset:X.shape[1] + y_offset, :] = X\n    return newX\n\ndef createImageCubes(X, y, windowSize=5, removeZeroLabels = True):\n    margin = int((windowSize - 1) / 2)\n    zeroPaddedX = padWithZeros(X, margin=margin)\n    # split patches\n    patchesData = np.zeros((X.shape[0] * X.shape[1], windowSize, windowSize, X.shape[2])).astype("float32")\n    patchesLabels = np.zeros((X.shape[0] * X.shape[1]))\n    patchIndex = 0\n    for r in range(margin, zeroPaddedX.shape[0] - margin):\n        for c in range(margin, zeroPaddedX.shape[1] - margin):\n            patch = zeroPaddedX[r - margin:r + margin + 1, c - margin:c + margin + 1]\n            patchesData[patchIndex, :, :, :] = patch.astype("float32")\n            patchesLabels[patchIndex] = y[r-margin, c-margin]\n            patchIndex = patchIndex + 1\n    if removeZeroLabels:\n        patchesData = patchesData[patchesLabels>0,:,:,:]\n        patchesLabels = patchesLabels[patchesLabels>0]\n        patchesLabels -= 1\n    return patchesData, patchesLabels.astype("int")\n\n\ndef accuracy(output, target, topk=(1,)):\n    output = np.argmax(output, axis=1)\n    return np.sum(output == target) * 100\n\n\ndef AA_andEachClassAccuracy(confusion_matrix):\n    counter = confusion_matrix.shape[0]\n    list_diag = np.diag(confusion_matrix)\n    list_raw_sum = np.sum(confusion_matrix, axis=1)\n    each_acc = np.nan_to_num(truediv(list_diag, list_raw_sum))\n    average_acc = np.mean(each_acc)\n    return each_acc, average_acc\n\n\ndef reports(y_pred, y_test):\n    classification = classification_report(y_test, y_pred)\n    oa = accuracy_score(y_test, y_pred)\n    confusion = confusion_matrix(y_test, y_pred)\n    each_acc, aa = AA_andEachClassAccuracy(confusion)\n    kappa = cohen_kappa_score(y_test, y_pred)\n\n    return classification, confusion, list(np.round(np.array([oa, aa, kappa] + list(each_acc)) * 100, 2))\n', 'dense_suppli.py': "# For running in python 2.x\nfrom __future__ import print_function, unicode_literals\nfrom __future__ import absolute_import, division\n\nimport sys\nimport numpy as np\nfrom sklearn.metrics import confusion_matrix\nfrom scipy.spatial.distance import cdist\nfrom keras.utils.np_utils import to_categorical\n\ndef relabel(labelTr, labelTs):\n    unqLab, pInClass=np.unique(labelTr, return_counts=True)\n    sortedUnqLab=np.argsort(pInClass, kind='mergesort')\n    c=sortedUnqLab.shape[0]\n    labelsNewTr=np.zeros((labelTr.shape[0],))-1\n    labelsNewTs=np.zeros((labelTs.shape[0],))-1\n    pInClass=np.sort(pInClass)\n    classMap=list()\n    for i in range(c):\n        labelsNewTr[labelTr==unqLab[sortedUnqLab[i]]]=i\n        labelsNewTs[labelTs==unqLab[sortedUnqLab[i]]]=i\n        classMap.append(np.where(labelsNewTr==i)[0])\n    return labelsNewTr, labelsNewTs, c, pInClass, classMap, sortedUnqLab\n\ndef irFind(pInClass, c, irIgnore=1):\n    ir=pInClass[-1]/pInClass\n    imbalancedCls=np.arange(c)[ir>irIgnore]\n    toBalance=np.subtract(pInClass[-1], pInClass[imbalancedCls])\n    imbClsNum=toBalance.shape[0]\n    if imbClsNum==0: sys.exit('No imbalanced classes found, exiting ...')\n    return imbalancedCls, toBalance, imbClsNum, ir\n\ndef fileRead(fileName):\n    dataTotal=np.loadtxt(fileName, delimiter=',')\n    data=dataTotal[:, :-1]\n    labels=dataTotal[:, -1]\n    return data, labels\n\ndef indices(pLabel, tLabel):\n    confMat=confusion_matrix(tLabel, pLabel)\n    nc=np.sum(confMat, axis=1)\n    tp=np.diagonal(confMat)\n    tpr=tp/nc\n    acsa=np.mean(tpr)\n    gm=np.prod(tpr)**(1/confMat.shape[0])\n    acc=np.sum(tp)/np.sum(nc)\n    return acsa, gm, tpr, confMat, acc\n\ndef randomLabelGen(toBalance, batchSize, c):\n    cumProb=np.cumsum(toBalance/np.sum(toBalance))\n    bins=np.insert(cumProb, 0, 0)\n    randomValue=np.random.rand(batchSize,)\n    randLabel=np.digitize(randomValue, bins)-1\n    randLabel_cat=to_categorical(randLabel)\n    labelPadding=np.zeros((batchSize, c-randLabel_cat.shape[1]))\n    randLabel_cat=np.hstack((randLabel_cat, labelPadding))\n    return randLabel_cat\n\ndef batchDivision(n, batchSize):\n    numBatches, residual=int(np.ceil(n/batchSize)), int(n%batchSize)\n    if residual==0:\n        residual=batchSize\n    batchDiv=np.zeros((numBatches+1,1), dtype='int64')\n    batchSizeStore=np.ones((numBatches, 1), dtype='int64')\n    batchSizeStore[0:-1, 0]=batchSize\n    batchSizeStore[-1, 0]=residual\n    for i in range(numBatches):\n        batchDiv[i]=i*batchSize\n    batchDiv[numBatches]=batchDiv[numBatches-1]+residual\n    return batchDiv, numBatches, batchSizeStore\n\ndef rearrange(labelsCat, numImbCls):\n    labels=np.argmax(labelsCat, axis=1)\n    arrangeMap=list()\n    for i in range(numImbCls):\n        arrangeMap.append(np.where(labels==i)[0])\n    return arrangeMap\n", 'README.md': '# Generative Adversarial Minority Oversampling for Spectral-Spatial Hyperspectral Image Classification\nThe Code for "Generative Adversarial Minority Oversampling for Spectral-Spatial Hyperspectral Image Classification". [https://ieeexplore.ieee.org/document/9347550]\n```\nS. K. Roy, J. M. Haut, M. E. Paoletti, S. R. Dubey and A. Plaza. \nGenerative Adversarial Minority Oversampling for Spectral-Spatial Hyperspectral Image Classification\nIEEE Transactions on Geoscience and Remote Sensing\nDOI: 10.1109/TGRS.2021.3052048\nFebruary 2021.\n```\n\n![3DGAMO](./images/3DGAMO.png)\n\n\n### Install and activate requires packages (with conda)\n```\nconda env create -f enviroment.yml\nconda activate 3D-HyperGAMO\n```\n\n### Example of use\n```\n# Without datasets\ngit clone https://github.com/mhaut/3D-HyperGAMO/\n\n# With datasets\ngit clone --recursive https://github.com/mhaut/3D-HyperGAMO/\ncd HSI-datasets\npython join_dsets.py\n```\n\n\n### Run code\n\n```\npython main.py --dataset IP \n\n```\n\nReference code: https://github.com/SankhaSubhra/GAMO\n'}


In [ ]:
WORKER_SOURCE = '"""3D-HyperGAMO training adapter: original networks, MSE and adversarial update sequence.\n\nOnly data protocol, compatibility, logging, validation selection and persistence differ.\nNever import the test cache in the training stage.\n"""\nimport argparse\nimport csv\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport traceback\nimport zipfile\n\nparser = argparse.ArgumentParser()\nparser.add_argument(\'stage\', choices=[\'smoke\', \'train\', \'samples\', \'evaluate\', \'infer\'])\nparser.add_argument(\'--run\', required=True)\nparser.add_argument(\'--input\', help=\'Prepared inference patches; no fitting or labels.\')\nparser.add_argument(\'--output\', help=\'Destination for inference probabilities.\')\nparser.add_argument(\'--cpu\', action=\'store_true\', help=\'CPU only for engineering smoke tests.\')\nargs = parser.parse_args()\nRUN = Path(args.run).resolve()\nCFG = json.loads((RUN / \'config.json\').read_text(encoding=\'utf-8\'))\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = \'-1\' if args.cpu else str(CFG[\'gpu_index\'])\nos.environ[\'TF_CPP_MIN_LOG_LEVEL\'] = \'2\'\nos.environ[\'TF_USE_LEGACY_KERAS\'] = \'1\'\nos.environ[\'PYTHONUNBUFFERED\'] = \'1\'\n\nimport numpy as np\nimport tensorflow as tf\nimport tf_keras as keras\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score, cohen_kappa_score, precision_recall_fscore_support\n\ntf.config.threading.set_intra_op_parallelism_threads(2)\ntf.config.threading.set_inter_op_parallelism_threads(2)\ndevices = tf.config.list_physical_devices(\'GPU\')\nfor dev in devices:\n    tf.config.experimental.set_memory_growth(dev, True)\nif args.stage != \'smoke\' and (not devices or args.cpu):\n    raise RuntimeError(\'Huấn luyện/đánh giá cần GPU. Chưa khởi chạy CPU thay thế.\')\nprint(\'TensorFlow:\', tf.__version__, \'| tf-keras:\', keras.__version__, \'| GPU:\', devices, flush=True)\nkeras.utils.set_random_seed(int(CFG[\'train_seed\']))\ntry:\n    tf.config.experimental.enable_op_determinism()\nexcept AttributeError:\n    pass\n\n# The pinned author source is embedded next to this adapter. Import-only compatibility changes.\nauthor_source = (RUN / \'code\' / \'dense_net3D_SA_original.py\').read_text(encoding=\'utf-8\')\nsource = author_source.replace(\'from keras.layers.advanced_activations import\', \'from tf_keras.layers import\')\nsource = source.replace(\'from keras\', \'from tf_keras\')\nNT = {}\nexec(compile(source, \'dense_net3D_SA_compat.py\', \'exec\'), NT)\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef bundle():\n    target = RUN.parent / (RUN.name + \'.zip\')\n    tmp = target.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(tmp, \'w\', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:\n        for path in sorted(RUN.rglob(\'*\')):\n            if path.is_file() and not path.name.endswith(\'.tmp\'):\n                archive.write(path, str(Path(RUN.name) / path.relative_to(RUN)))\n    tmp.replace(target)\n    print(\'ZIP:\', target, flush=True)\n\n\ndef make_models(x, y, seed):\n    keras.utils.set_random_seed(seed)\n    np.random.seed(seed)\n    ids, counts = np.unique(y, return_counts=True)\n    assert np.array_equal(ids, np.arange(len(ids)))\n    order = np.argsort(counts, kind=\'mergesort\')\n    inverse = np.empty_like(order)\n    inverse[order] = np.arange(len(order))\n    y = inverse[y].astype(np.int64)\n    counts = counts[order]\n    permutation = np.random.choice(np.arange(len(x)), size=len(x), replace=False)\n    x, y = x[permutation], y[permutation]\n    c = len(order)\n    minority = np.flatnonzero(counts < counts[-1])\n    if len(minority) == 0:\n        raise RuntimeError(\'Train đã cân bằng; không có lớp thiểu số cho HyperGAMO.\')\n    assert np.array_equal(minority, np.arange(len(minority)))\n    supports = [x[y == i] for i in minority]\n    shape = tuple(x.shape[1:])\n    # Original code uses ONE shared Adam across C, D and composite G models.\n    opt = keras.optimizers.legacy.Adam(learning_rate=0.0002, beta_1=0.5)\n    C = NT[\'denseMlpCreate\'](sh=shape, num_class=c)\n    C.compile(loss=\'mean_squared_error\', optimizer=opt)\n    C.trainable = False\n    D = NT[\'denseDisCreate\'](shape, num_class=c)\n    D.compile(loss=\'mean_squared_error\', optimizer=opt)\n    D.trainable = False\n    G = NT[\'denseGamoGenCreate\'](100, c)\n    heads, vs_c, vs_d = [], [], []\n    for support in supports:\n        head = NT[\'denseGenProcessCreate\'](len(support), support, sh=shape, mul=int(np.prod(shape)))\n        heads.append(head)\n        z = keras.layers.Input(shape=(100,))\n        label = keras.layers.Input(shape=(c,))\n        mixed = head(G([z, label]))\n        g_c = keras.Model([z, label], C(mixed))\n        g_c.compile(loss=\'mean_squared_error\', optimizer=opt)\n        vs_c.append(g_c)\n        z2 = keras.layers.Input(shape=(100,))\n        label2 = keras.layers.Input(shape=(c,))\n        label3 = keras.layers.Input(shape=(c,))\n        mixed2 = head(G([z2, label2]))\n        g_d = keras.Model([z2, label2, label3], D([mixed2, label3]))\n        g_d.compile(loss=\'mean_squared_error\', optimizer=opt)\n        vs_d.append(g_d)\n    tracked = tf.Module()\n    tracked.classifier, tracked.discriminator, tracked.generator = C, D, G\n    tracked.optimizer = opt\n    for i, head in enumerate(heads):\n        setattr(tracked, f\'head_{i:03d}\', head)\n    checkpoint = tf.train.Checkpoint(models=tracked)\n    return dict(C=C, D=D, G=G, heads=heads, vs_c=vs_c, vs_d=vs_d, checkpoint=checkpoint,\n                optimizer=opt, x=x, y=y, order=order, inverse=inverse, counts=counts,\n                supports=supports, shape=shape, c=c)\n\n\ndef scalar(value):\n    value = float(np.asarray(value).reshape(-1)[0])\n    if not np.isfinite(value):\n        raise FloatingPointError(\'Loss NaN/Inf. Dừng, không ghi kết quả thành công.\')\n    return value\n\n\ndef predict(C, x, batch=32):\n    return np.concatenate([C.predict_on_batch(x[i:i+batch]) for i in range(0, len(x), batch)])\n\n\ndef metrics(y, pred, c):\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y, pred, labels=np.arange(c), zero_division=0)\n    result = dict(OA=float(accuracy_score(y, pred)), AA=float(recall.mean()),\n                  Macro_F1=float(f1.mean()), GM=float(np.exp(np.log(recall).mean())) if (recall > 0).all() else 0.,\n                  Kappa=float(cohen_kappa_score(y, pred)))\n    return result, (precision, recall, f1, support)\n\n\ndef snap(model):\n    return [w.numpy().copy() for w in model.weights]\n\n\ndef changed(model, before):\n    return any(not np.array_equal(w.numpy(), old) for w, old in zip(model.weights, before))\n\n\ndef smoke():\n    # Same spatial/spectral dimensions as the intended run; three artificial classes only.\n    shape = (CFG[\'patch_size\'], CFG[\'patch_size\'], CFG[\'pca_components\'], 1)\n    rng = np.random.default_rng(7)\n    x = rng.normal(size=(9, *shape)).astype(\'float32\')\n    y = np.repeat(np.arange(3), [2, 3, 4])\n    m = make_models(x, y, 7)\n    real = m[\'supports\'][0][:2]\n    labels = keras.utils.to_categorical([0, 0], 3).astype(\'float32\')\n    noise = rng.normal(size=(2, 100)).astype(\'float32\')\n    for name, model, inputs, targets in [\n        (\'C real\', m[\'C\'], real, labels),\n        (\'D real\', m[\'D\'], [real, labels], np.ones((2, 1), \'float32\')),\n    ]:\n        before = snap(model)\n        loss = scalar(model.train_on_batch(inputs, targets))\n        assert changed(model, before), name + \': trọng số không cập nhật.\'\n        print(\'PASS\', name, \'loss\', loss, flush=True)\n    fake = m[\'heads\'][0].predict_on_batch(m[\'G\'].predict_on_batch([noise, labels]))\n    assert fake.shape == real.shape and np.isfinite(fake).all()\n    assert (fake >= real.min(0) - 1e-5).all() and (fake <= real.max(0) + 1e-5).all()\n    for name, model, inputs, targets in [\n        (\'C fake\', m[\'C\'], fake, labels),\n        (\'D fake\', m[\'D\'], [fake, labels], np.zeros((2, 1), \'float32\')),\n    ]:\n        before = snap(model)\n        loss = scalar(model.train_on_batch(inputs, targets))\n        assert changed(model, before), name + \': trọng số không cập nhật.\'\n        print(\'PASS\', name, \'loss\', loss, flush=True)\n    for name, model, inputs, targets in [\n        (\'G against C\', m[\'vs_c\'][0], [noise, labels], 1-labels),\n        (\'G against D\', m[\'vs_d\'][0], [noise, labels, labels], np.ones((2, 1), \'float32\')),\n    ]:\n        cb, db, gb, hb = map(snap, [m[\'C\'], m[\'D\'], m[\'G\'], m[\'heads\'][0]])\n        loss = scalar(model.train_on_batch(inputs, targets))\n        assert not changed(m[\'C\'], cb) and not changed(m[\'D\'], db), name + \': C/D không đóng băng.\'\n        assert changed(m[\'G\'], gb) and changed(m[\'heads\'][0], hb), name + \': G/head không cập nhật.\'\n        print(\'PASS\', name, \'loss\', loss, flush=True)\n    # Checkpoint reload must preserve model output before any long run.\n    smoke_dir = RUN / \'smoke_checkpoint\'\n    smoke_dir.mkdir(exist_ok=True)\n    path = m[\'checkpoint\'].write(str(smoke_dir / \'check\'))\n    expected = m[\'C\'].predict_on_batch(real)\n    first = m[\'C\'].weights[0]\n    first.assign(tf.zeros_like(first))\n    m[\'checkpoint\'].restore(path).expect_partial()\n    np.testing.assert_allclose(m[\'C\'].predict_on_batch(real), expected, rtol=1e-5, atol=1e-6)\n    atomic_json(RUN / \'smoke_pass.json\', dict(passed=True, tensorflow=tf.__version__, tf_keras=keras.__version__,\n                shape=list(shape), gpu=bool(devices), data_is_artificial=True))\n    print(\'SMOKE PASSED — đây không phải kết quả nghiên cứu.\', flush=True)\n\n\ndef write_history(history):\n    path = RUN / \'training_history.csv\'\n    tmp = path.with_suffix(\'.csv.tmp\')\n    with open(tmp, \'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(history[0]))\n        writer.writeheader()\n        writer.writerows(history)\n    tmp.replace(path)\n\n\ndef train():\n    started = time.monotonic()\n    if not (RUN / \'smoke_pass.json\').exists():\n        raise RuntimeError(\'Chưa có smoke test thành công.\')\n    state_path = RUN / \'training_state.json\'\n    old_state = json.loads(state_path.read_text()) if state_path.exists() else None\n    if old_state and old_state[\'complete\']:\n        print(\'Lần chạy này đã hoàn thành. Không train lại.\', flush=True)\n        return\n    data = np.load(RUN / \'train_val.npz\', allow_pickle=False)\n    x, y = data[\'X_train\'], data[\'y_train\']\n    val_x, val_y_orig = data[\'X_val\'], data[\'y_val\']\n    m = make_models(x, y, int(CFG[\'train_seed\']))\n    val_y = m[\'inverse\'][val_y_orig]\n    class_ids = np.asarray(CFG[\'accepted_class_ids\'], dtype=int)[m[\'order\']]\n    atomic_json(RUN / \'model_class_order.json\', dict(original_class_ids=class_ids.tolist()))\n    latest = tf.train.CheckpointManager(m[\'checkpoint\'], str(RUN / \'checkpoints\' / \'latest\'), max_to_keep=2)\n    best = tf.train.CheckpointManager(m[\'checkpoint\'], str(RUN / \'checkpoints\' / \'best\'), max_to_keep=2)\n    use_val = CFG[\'protocol\'] == \'dip_train_only_validation\'\n    if use_val != bool(len(val_x)):\n        raise RuntimeError(\'Cache validation không khớp protocol.\')\n    step, history, best_f1 = 0, [], -1.\n    generated_counts = np.zeros(m[\'c\'], dtype=np.int64)\n    # Roll back metadata too if a prior process died between saving best and latest.\n    if old_state and old_state.get(\'selection\'):\n        atomic_json(RUN / \'selection.json\', old_state[\'selection\'])\n    if (RUN / \'selection.json\').exists():\n        best_f1 = json.loads((RUN / \'selection.json\').read_text()).get(\'Macro_F1\', -1.)\n    if old_state:\n        if not CFG[\'resume_training\']:\n            raise RuntimeError(\'Có checkpoint dở dang. Bật resume_training để tiếp tục, không ghi đè.\')\n        path = RUN / old_state[\'checkpoint\']\n        m[\'checkpoint\'].restore(str(path)).expect_partial()\n        step = int(old_state[\'step\'])\n        generated_counts = np.array(old_state[\'generated_counts\'], dtype=np.int64)\n        state = old_state[\'numpy_rng\']\n        np.random.set_state((state[0], np.asarray(state[1], dtype=np.uint32), state[2], state[3], state[4]))\n        if (RUN / \'training_history.csv\').exists():\n            with open(RUN / \'training_history.csv\', newline=\'\') as f:\n                history = [dict(row) for row in csv.DictReader(f) if int(row[\'step\']) <= step]\n        print(\'Tiếp tục từ step\', step, \'(không cam kết trùng bit với chạy liên tục: RNG Dropout).\', flush=True)\n    labels = keras.utils.to_categorical(m[\'y\'], m[\'c\']).astype(\'float32\')\n    n, bs = len(m[\'x\']), int(CFG[\'batch_size\'])\n    n_batches = int(math.ceil(n / bs))\n    gen_points = int(math.ceil(bs / m[\'c\']))\n    deficits = m[\'counts\'][-1] - m[\'counts\'][:len(m[\'heads\'])]\n    cumulative = np.cumsum(deficits / deficits.sum())\n    max_step = int(CFG[\'max_step\'])\n    if max_step <= 0 or max_step % 2:\n        raise ValueError(\'max_step phải là số chẵn dương (mỗi vòng có hai cập nhật C).\')\n    recent = np.zeros(4, dtype=float)\n    recent_n = 0\n    while step < max_step:\n        j = (step // 2) % n_batches\n        a, b = j * bs, min((j + 1) * bs, n)\n        size = b - a\n        valid_r = 1. - np.random.uniform(0., .1, (size, 1))\n        loss_c_real = scalar(m[\'C\'].train_on_batch(m[\'x\'][a:b], labels[a:b]))\n        loss_d_real = scalar(m[\'D\'].train_on_batch([m[\'x\'][a:b], labels[a:b]], valid_r))\n        invalid = np.random.uniform(0., .1, (size, 1))\n        z = np.random.normal(size=(size, 100))\n        fake_ids = np.searchsorted(cumulative, np.random.rand(size), side=\'right\')\n        generated_counts += np.bincount(fake_ids, minlength=m[\'c\'])\n        fake_labels = keras.utils.to_categorical(fake_ids, m[\'c\']).astype(\'float32\')\n        features = m[\'G\'].predict_on_batch([z, fake_labels])\n        fake = np.zeros((size, *m[\'shape\']), dtype=\'float32\')\n        for i, head in enumerate(m[\'heads\']):\n            idx = np.flatnonzero(fake_ids == i)\n            if len(idx):\n                fake[idx] = head.predict_on_batch(features[idx])\n        loss_c_fake = scalar(m[\'C\'].train_on_batch(fake, fake_labels))\n        loss_d_fake = scalar(m[\'D\'].train_on_batch([fake, fake_labels], invalid))\n        g_c, g_d = [], []\n        for i in range(len(m[\'heads\'])):\n            condition = np.zeros((gen_points, m[\'c\']), dtype=\'float32\')\n            condition[:, i] = 1\n            noise = np.random.normal(size=(gen_points, 100))\n            # Important: retain the original adversarial target, NOT normal class CE.\n            g_c.append(scalar(m[\'vs_c\'][i].train_on_batch([noise, condition], 1-condition)))\n            g_d.append(scalar(m[\'vs_d\'][i].train_on_batch(\n                [noise, condition, condition], np.ones((gen_points, 1)))))\n        recent += [(loss_c_real + loss_c_fake) / 2, (loss_d_real + loss_d_fake) / 2,\n                   np.mean(g_c), np.mean(g_d)]\n        recent_n += 1\n        step += 2\n        timed_out = (time.monotonic() - started) / 60 >= CFG[\'max_minutes\']\n        if step % int(CFG[\'eval_every\']) == 0 or step == max_step or timed_out:\n            values = {}\n            if use_val:\n                val_pred = predict(m[\'C\'], val_x, CFG[\'eval_batch_size\']).argmax(1)\n                values, _ = metrics(val_y, val_pred, m[\'c\'])\n            row = dict(step=step, **values, C_loss=recent[0]/recent_n, D_loss=recent[1]/recent_n,\n                       G_vs_C_loss=recent[2]/recent_n, G_vs_D_loss=recent[3]/recent_n,\n                       elapsed_minutes=(time.monotonic()-started)/60)\n            history.append(row)\n            write_history(history)\n            if (use_val and values[\'Macro_F1\'] > best_f1) or (not use_val and step == max_step):\n                best_f1 = values.get(\'Macro_F1\', -1.)\n                best_path = best.save(checkpoint_number=step)\n                # The metadata must not point to an unfinished/mismatched classifier file.\n                classifier_path = RUN / \'checkpoints\' / \'best\' / f\'classifier_{step}.weights.h5\'\n                temporary = classifier_path.with_name(f\'classifier_{step}.tmp.weights.h5\')\n                m[\'C\'].save_weights(str(temporary))\n                temporary.replace(classifier_path)\n                atomic_json(RUN / \'selection.json\', dict(\n                    step=step, rule=\'validation_macro_f1\' if use_val else \'fixed_final_budget\',\n                    **values, checkpoint=str(Path(best_path).relative_to(RUN)),\n                    classifier_weights=str(classifier_path.relative_to(RUN))))\n            last_path = latest.save(checkpoint_number=step)\n            rng = np.random.get_state()\n            state = dict(step=step, complete=step >= max_step, checkpoint=str(Path(last_path).relative_to(RUN)),\n                         numpy_rng=[rng[0], rng[1].tolist(), int(rng[2]), int(rng[3]), float(rng[4])],\n                         model_class_ids=class_ids.tolist(), generated_counts=generated_counts.tolist(),\n                         selection=json.loads((RUN / \'selection.json\').read_text()) if (RUN / \'selection.json\').exists() else None,\n                         reason=\'time_limit\' if timed_out and step < max_step else \'checkpoint\')\n            atomic_json(state_path, state)\n            atomic_json(RUN / \'online_generated_counts.json\', dict(class_ids=class_ids.tolist(),\n                real_train_counts=m[\'counts\'].tolist(), generated_presentations=generated_counts.tolist(),\n                step=step, note=\'Cumulative fake examples presented to C, not unique saved samples.\'))\n            print(\'VALIDATION\' if use_val else \'TRAIN\', json.dumps({k: v for k, v in row.items() if k not in [\'elapsed_minutes\']}, allow_nan=False), flush=True)\n            bundle()\n            recent[:] = 0\n            recent_n = 0\n        if timed_out and step < max_step:\n            print(\'Hết thời gian đã đặt. Đã lưu checkpoint và ZIP; CHƯA đánh giá test.\', flush=True)\n            return\n    print(\'TRAIN COMPLETE. Chọn checkpoint:\', \'Macro-F1 validation\' if use_val else \'bước cuối cố định\', flush=True)\n\n\ndef evaluate():\n    state = json.loads((RUN / \'training_state.json\').read_text())\n    if not state[\'complete\']:\n        raise RuntimeError(\'Train chưa đủ ngân sách đã đặt; không tự đánh giá test.\')\n    if (RUN / \'test_complete.json\').exists():\n        for name in [\'test_metrics.json\', \'test_per_class.csv\', \'test_predictions.npz\']:\n            if not (RUN / name).is_file():\n                raise RuntimeError(\'Kết quả test thiếu file: \' + name)\n        print(\'Đã có test cuối của run này. Không đánh giá lại.\', flush=True)\n        return\n    # Build only the classifier and load protocol-selected weights.\n    order = json.loads((RUN / \'model_class_order.json\').read_text())[\'original_class_ids\']\n    shape = (CFG[\'patch_size\'], CFG[\'patch_size\'], CFG[\'pca_components\'], 1)\n    C = NT[\'denseMlpCreate\'](shape, len(order))\n    selected = json.loads((RUN / \'selection.json\').read_text())\n    C.load_weights(str(RUN / selected[\'classifier_weights\']))\n    test = np.load(RUN / \'test_locked.npz\', allow_pickle=False)\n    x, original_y = test[\'X_test\'], test[\'original_class_ids\']\n    lookup = {int(cid): i for i, cid in enumerate(order)}\n    y = np.array([lookup[int(cid)] for cid in original_y], dtype=np.int64)\n    prob = predict(C, x, CFG[\'eval_batch_size\'])\n    pred = prob.argmax(1)\n    values, (precision, recall, f1, support) = metrics(y, pred, len(order))\n    atomic_json(RUN / \'test_metrics.json\', dict(**values, selected_step=selected[\'step\'],\n                evaluated_classes=order, protocol=CFG[\'protocol\'], selection_rule=selected[\'rule\'], patch_overlap_expected=True, seed=CFG[\'train_seed\']))\n    with open(RUN / \'test_per_class.csv\', \'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.writer(f)\n        writer.writerow([\'class_id\', \'precision\', \'recall\', \'f1\', \'test_support\'])\n        for i, cid in sorted(enumerate(order), key=lambda item: item[1]):\n            writer.writerow([cid, precision[i], recall[i], f1[i], int(support[i])])\n    np.savez_compressed(RUN / \'test_predictions.npz\', coords=test[\'coords_test\'],\n                        true_class_ids=original_y, predicted_class_ids=np.asarray(order)[pred],\n                        probabilities=prob, probability_class_ids=np.asarray(order),\n                        confusion_matrix=confusion_matrix(y, pred, labels=np.arange(len(order))))\n    atomic_json(RUN / \'test_complete.json\', dict(complete=True, selected_step=selected[\'step\']))\n    print(\'FINAL TEST — DIP, đầy đủ các lớp đã cấu hình; xem audit chồng lấn:\', values, flush=True)\n\n\ndef samples():\n    """Diagnostics use training supports only; no filtering or quality claim is implied."""\n    state = json.loads((RUN / \'training_state.json\').read_text())\n    if not state[\'complete\']:\n        raise RuntimeError(\'Chưa hoàn thành huấn luyện.\')\n    if (RUN / \'samples_complete.json\').exists():\n        for name in [\'generated_examples.npz\', \'generation_diagnostics.csv\']:\n            if not (RUN / name).is_file():\n                raise RuntimeError(\'Minh họa mẫu sinh thiếu file: \' + name)\n        print(\'Đã có mẫu minh họa của checkpoint được chọn.\', flush=True)\n        return\n    cache = np.load(RUN / \'train_val.npz\', allow_pickle=False)\n    m = make_models(cache[\'X_train\'], cache[\'y_train\'], int(CFG[\'train_seed\']))\n    selection = json.loads((RUN / \'selection.json\').read_text())\n    m[\'checkpoint\'].restore(str(RUN / selection[\'checkpoint\'])).expect_partial()\n    ids = np.asarray(CFG[\'accepted_class_ids\'])[m[\'order\']]\n    rng = np.random.default_rng(1701)\n    examples, real_examples, sample_ids, rows, generated_probs = [], [], [], [], []\n    from scipy.spatial.distance import cdist\n    for i, head in enumerate(m[\'heads\']):\n        condition = np.zeros((16, m[\'c\']), dtype=\'float32\')\n        condition[:, i] = 1\n        feature = m[\'G\'].predict_on_batch([rng.normal(size=(16, 100)), condition])\n        fake = head.predict_on_batch(feature)\n        generated_probs.append(predict(m[\'C\'], fake, CFG[\'eval_batch_size\']))\n        weights_model = keras.Model(head.inputs, head.layers[1].output)\n        weights = weights_model.predict_on_batch(feature)\n        h = -(weights * np.log(np.maximum(weights, 1e-12))).sum(axis=1)\n        support = m[\'supports\'][i]\n        distance = cdist(fake.reshape(16, -1), support.reshape(len(support), -1), metric=\'euclidean\')\n        nearest = distance.argmin(axis=1)\n        real_examples.append(support[nearest])\n        examples.append(fake)\n        sample_ids.extend([int(ids[i])] * len(fake))\n        rows.append(dict(class_id=int(ids[i]), real_train=len(support),\n                         normalized_entropy=float((h / np.log(len(support))).mean()),\n                         effective_supports=float(np.exp(h).mean()),\n                         mean_max_weight=float(weights.max(axis=1).mean()),\n                         nearest_real_RMSE=float((distance.min(axis=1) / np.sqrt(np.prod(m[\'shape\']))).mean()),\n                         variance_ratio=float(fake.var(axis=0).mean() / (support.var(axis=0).mean() + 1e-12))))\n    np.savez_compressed(RUN / \'generated_examples.npz\', fake=np.concatenate(examples),\n                        nearest_real=np.concatenate(real_examples), original_class_ids=np.asarray(sample_ids),\n                        probabilities=np.concatenate(generated_probs), probability_class_ids=ids)\n    with open(RUN / \'generation_diagnostics.csv\', \'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    atomic_json(RUN / \'samples_complete.json\', dict(complete=True, selected_step=selection[\'step\']))\n    print(\'Đã lưu mẫu và chẩn đoán theo lớp; chưa phải bằng chứng GAN cải thiện phân loại.\', flush=True)\n\n\n\ndef infer():\n    """Predict user-supplied preprocessed patches with the selected classifier."""\n    if not (RUN / \'test_complete.json\').exists():\n        raise RuntimeError(\'Hoàn thành test cố định trước khi dùng ứng dụng minh họa.\')\n    if not args.input or not args.output:\n        raise ValueError(\'Cần --input và --output cho infer.\')\n    destination = Path(args.output).resolve()\n    if not destination.is_relative_to(RUN):\n        raise ValueError(\'Đầu ra ứng dụng phải nằm trong run.\')\n    x = np.load(args.input, allow_pickle=False)[\'patches\']\n    shape = (CFG[\'patch_size\'], CFG[\'patch_size\'], CFG[\'pca_components\'], 1)\n    if tuple(x.shape[1:]) != shape or len(x) == 0 or not np.isfinite(x).all():\n        raise ValueError(\'Shape hoặc dữ liệu ứng dụng không hợp lệ.\')\n    order = json.loads((RUN / \'model_class_order.json\').read_text())[\'original_class_ids\']\n    C = NT[\'denseMlpCreate\'](shape, len(order))\n    selected = json.loads((RUN / \'selection.json\').read_text())\n    C.load_weights(str(RUN / selected[\'classifier_weights\']))\n    prob = predict(C, x, CFG[\'eval_batch_size\'])\n    np.savez_compressed(destination, probabilities=prob, class_ids=np.asarray(order))\n    print(\'Ứng dụng: đã phân lớp\', len(x), \'vùng ảnh; không cộng vào độ chính xác test.\', flush=True)\n\n\ntry:\n    if args.stage == \'smoke\':\n        smoke()\n    elif args.stage == \'train\':\n        train()\n    elif args.stage == \'samples\':\n        samples()\n    elif args.stage == \'evaluate\':\n        evaluate()\n    else:\n        infer()\nexcept Exception:\n    (RUN / f\'error_{args.stage}.txt\').write_text(traceback.format_exc(), encoding=\'utf-8\')\n    raise\nfinally:\n    # Persist partial results even if a normal Python exception occurs.\n    # A forcibly killed Kaggle VM cannot run this finally block: Save Version is still required.\n    bundle()\n'
compile(WORKER_SOURCE, 'hypergamo_worker.py', 'exec')
print('Đã nạp source; chưa khởi tạo GPU.')


## 7. Tạo thư mục chạy riêng và lưu nguồn gốc

Cấu hình, mã, dữ liệu, cách chia và môi trường số được ghi dấu vân tay. Hai chế độ có
thư mục kết quả khác nhau. Không xóa output cũ. ZIP tiếp tục phải khớp đúng lần chạy.
Chỉ mở checkpoint/joblib từ ZIP do chính bạn tạo vì đây không phải định dạng an toàn
để nhận từ nguồn không tin cậy.


In [ ]:
# 8. THƯ MỤC RIÊNG CÓ ĐỊNH DANH — KHÔNG TRỘN KẾT QUẢ CŨ
identity = {
    # Runtime controls may change when resuming; model/split/budget must not change.
    'config': {k: v for k, v in CFG.items() if k not in ['max_minutes', 'gpu_index', 'resume_training']},
    'source_data': source_hashes,
    'dip_sha256': hashlib.sha256(dip_bytes).hexdigest(),
    'numeric_versions': {p: importlib.metadata.version(p) for p in ['numpy', 'scipy', 'scikit-learn', 'tensorflow', 'tf-keras']},
    'author_commit': '3f92889aec4a7e2c8f7a2db4c8a9bf167f296f5e',
    'networks_sha256': hashlib.sha256(AUTHOR_NETWORKS.encode()).hexdigest(),
    'worker_sha256': hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(),
    'data_code_sha256': hashlib.sha256(DATA_SOURCE.encode()).hexdigest(),
    'split_sha256': hashlib.sha256(split.tobytes()).hexdigest(),
}
run_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:12]
RUN = OUTPUT_ROOT / f'hypergamo_16class_{CFG["protocol"]}_{run_id}'

if RESUME_ZIP:
    # Chỉ ZIP do notebook này tạo. Không thực thi mã lấy từ ZIP; mã hiện tại được ghi lại bên dưới.
    import zipfile
    if RUN.exists():
        raise RuntimeError('Đã có thư mục run. Đặt RESUME_ZIP=None để dùng dữ liệu đang có, tránh ghi đè.')
    with zipfile.ZipFile(RESUME_ZIP) as archive:
        for entry in archive.infolist():
            relative = Path(entry.filename)
            if relative.is_absolute() or '..' in relative.parts or relative.parts[0] != RUN.name:
                raise RuntimeError('ZIP sai cấu hình hoặc có đường dẫn không hợp lệ.')
            target = (OUTPUT_ROOT / relative).resolve()
            if not target.is_relative_to(RUN.resolve()):
                raise RuntimeError('Đường dẫn ngoài run.')
        archive.extractall(OUTPUT_ROOT)

RUN.mkdir(parents=True, exist_ok=True)
manifest_path = RUN / 'manifest.json'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert saved['identity'] == identity, 'Run khác cấu hình hoặc dữ liệu.'
else:
    manifest_path.write_text(json.dumps({'identity': identity, 'overlap': overlap_report,
        'label_audit': label_audit, 'excluded_classes': [],
        'note': '16-class author DIP; overlapping input patches; not a spatial leakage-free benchmark'},
        ensure_ascii=False, indent=2), encoding='utf-8')
(RUN / 'config.json').write_text(json.dumps(CFG, indent=2), encoding='utf-8')
(RUN / 'code').mkdir(exist_ok=True)
for name, source in [('dip_data.py', DATA_SOURCE), ('dense_net3D_SA_original.py', AUTHOR_NETWORKS),
                     ('main_original.py', AUTHOR_MAIN), ('hypergamo_worker.py', WORKER_SOURCE)]:
    (RUN / 'code' / name).write_text(source, encoding='utf-8')
for name, source in AUTHOR_REFERENCE_FILES.items():
    (RUN / 'code' / ('author_' + name)).write_text(source, encoding='utf-8')
(RUN / 'indianpines_disjoint_dset.mat').write_bytes(dip_bytes)
np.savez_compressed(RUN / 'split_indices.npz', split=split, labels_protocol=labels_protocol,
                    coords_train=coords_by_split[0], coords_val=coords_by_split[1], coords_test=coords_by_split[2],
                    standard_gt=gt, class_ids=np.arange(1, 17))
count_table.to_csv(RUN / 'split_counts.csv', index=False)
for filename, fig in FIGURES.items():
    fig.savefig(RUN / filename, dpi=140, bbox_inches='tight')
    plt.close(fig)
FIGURES.clear()
versions = {p: importlib.metadata.version(p) for p in ['tensorflow', 'tf-keras', 'numpy', 'scipy', 'scikit-learn', 'pandas', 'joblib']}
(RUN / 'environment.json').write_text(json.dumps({'python': platform.python_version(), **versions}, indent=2))
print('Run:', RUN)


## 8. PCA, chuẩn hóa và tạo vùng ảnh

PCA giảm 200 dải thành 40 **thành phần**, không gọi 40 thành phần này là 40 dải vật lý.
Chuẩn hóa từng thành phần về [-1,1] trên tập dùng fit. Giữ viền bằng đệm số 0 sau
tiền xử lý, như repo. Chỉ trích vùng ảnh cho các tâm cần dùng, không tạo toàn bộ bản sao.

Mặc định fit toàn cảnh để bám quy trình cũ: có sử dụng đặc trưng của vùng test dù không
dùng nhãn test để fit. Nhánh train-only chỉ fit trên phổ tại tâm train; điều này **không
loại bỏ việc các patch train nhìn thấy pixel thuộc vùng test**.


In [ ]:
train_coords = coords_by_split[0]
flat = hsi.reshape(-1, hsi.shape[-1]).astype('float64')
fit_spectra = flat if CFG['protocol'] == 'dip_reproduction' else hsi[train_coords[:, 0], train_coords[:, 1]].astype('float64')
if (RUN / 'preprocessing.joblib').exists():
    preprocessing = joblib.load(RUN / 'preprocessing.joblib')
    pca, scaler = preprocessing['pca'], preprocessing['scaler']
else:
    pca = PCA(n_components=CFG['pca_components'], svd_solver=CFG['pca_solver'])
    fit_features = pca.fit_transform(fit_spectra)
    scaler = MinMaxScaler(feature_range=(-1, 1)).fit(fit_features)
    preprocessing = dict(pca=pca, scaler=scaler, raw_bands=hsi.shape[-1],
                         patch_size=CFG['patch_size'], class_ids=CFG['accepted_class_ids'],
                         protocol=CFG['protocol'], fit_count=len(fit_spectra))
    joblib.dump(preprocessing, RUN / 'preprocessing.joblib')
assert pca.n_samples_ == len(fit_spectra)
np.testing.assert_allclose(pca.mean_, fit_spectra.mean(0), rtol=1e-6, atol=1e-5)
processed = scaler.transform(pca.transform(flat)).reshape(145, 145, CFG['pca_components']).astype('float32')
print('Fit PCA/scaler từ:', 'toàn cảnh' if CFG['protocol'] == 'dip_reproduction' else 'tâm train')
print('Số phổ dùng fit:', len(fit_spectra), '| Phương sai giữ lại:', pca.explained_variance_ratio_.sum())
print('Shape:', processed.shape, '| Miền giá trị:', processed.min(), processed.max())
print('Nhánh train-only có thể cho val/test ngoài [-1,1]; không fit lại theo test.')
del flat, fit_spectra


In [ ]:
# 10. TRÍCH VÙNG ẢNH, LƯU TRAIN/VAL RIÊNG VỚI TEST
class_to_index = {cid: i for i, cid in enumerate(CFG['accepted_class_ids'])}
prepared = RUN / 'data_ready.json'
if not prepared.exists():
    datasets = []
    for coords in coords_by_split:
        x = extract_patches(processed, coords, CFG['patch_size'])
        original_labels = labels_protocol[coords[:, 0], coords[:, 1]].astype('int64')
        y = np.array([class_to_index[int(c)] for c in original_labels], dtype='int64')
        datasets.append((x, y, original_labels))
    np.savez_compressed(RUN / 'train_val.npz', X_train=datasets[0][0], y_train=datasets[0][1],
                        X_val=datasets[1][0], y_val=datasets[1][1])
    np.savez_compressed(RUN / 'test_locked.npz', X_test=datasets[2][0],
                        original_class_ids=datasets[2][2], coords_test=coords_by_split[2])
    ready = {'shapes': [list(d[0].shape) for d in datasets],
             'train_val_sha256': digest_file(RUN / 'train_val.npz'),
             'test_sha256': digest_file(RUN / 'test_locked.npz'),
             'preprocessing_sha256': digest_file(RUN / 'preprocessing.joblib')}
    prepared.write_text(json.dumps(ready, indent=2))
    del datasets
ready = json.loads(prepared.read_text())
for filename, key in [('train_val.npz', 'train_val_sha256'), ('test_locked.npz', 'test_sha256'),
                      ('preprocessing.joblib', 'preprocessing_sha256')]:
    assert digest_file(RUN / filename) == ready[key], f'Cache thay đổi: {filename}'
print('Shape train/validation/test:', ready['shapes'])
del processed


In [ ]:
# 11. HÀM CHẠY VÀ ĐÓNG GÓI — không bắt đầu huấn luyện ở ô này
def zip_run():
    import zipfile
    target = RUN.parent / (RUN.name + '.zip')
    temporary = target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for path in sorted(RUN.rglob('*')):
            if path.is_file() and not path.name.endswith('.tmp'):
                archive.write(path, str(Path(RUN.name) / path.relative_to(RUN)))
    temporary.replace(target)
    return target

def run_stage(stage, extra_args=None):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['PYTHONIOENCODING'] = 'utf-8'
    command = [sys.executable, '-u', str(RUN / 'code' / 'hypergamo_worker.py'), stage, '--run', str(RUN), *(extra_args or [])]
    print('Bắt đầu:', stage, flush=True)
    log_path = RUN / f'{stage}.log'
    with open(log_path, 'a', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', env=env)
        try:
            for line in process.stdout:
                log.write(line); log.flush()
                print(line, end='', flush=True)
            status = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=20)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
        finally:
            archive_path = zip_run()
            print('Đã đóng gói:', archive_path, flush=True)
    if status != 0:
        raise RuntimeError(f'Bước {stage} lỗi. Đọc {log_path.name}; không tự chạy bước sau.')

print('Đã lưu dữ liệu và mã:', zip_run())


## 9. Kiểm tra kỹ thuật trước khi huấn luyện

Kiểm tra cùng kích thước 11×11×40: C/D thật sự cập nhật, C/D đóng băng khi G học,
G và đầu sinh cập nhật, mẫu sinh hữu hạn, nằm trong bao tổ hợp lồi; checkpoint nạp lại
cho cùng dự đoán. Dữ liệu nhỏ nhân tạo chỉ dùng kiểm tra phần mềm, **không đưa vào kết quả KLTN**.


In [ ]:
if RUN_TRAINING:
    if not (RUN / 'smoke_pass.json').exists():
        run_stage('smoke')
    print(json.loads((RUN / 'smoke_pass.json').read_text()))
else:
    print('RUN_TRAINING=False: chỉ chuẩn bị dữ liệu, chưa kiểm tra/huấn luyện GPU.')


## 10. Huấn luyện mô hình gốc

C thật → D thật → tạo giả theo thiếu hụt lớp → C giả → D giả → G–C và G–D từng lớp.
Mỗi vòng có hai cập nhật C, vì thế `max_step=8000` là 4000 vòng như cách đếm của repo.

Mỗi 500 bước ghi loss và checkpoint. Nhánh có validation đánh giá val, chọn Macro-F1;
nhánh mặc định không có val và dùng trọng số bước cuối. **Không đọc cache test khi train.**
Thời gian chạy tùy GPU; giới hạn 180 phút không bảo đảm đã đủ 8000 bước. Nếu chưa đủ,
notebook dừng có ZIP và không tự xuất test. Resume lưu optimizer và RNG NumPy nhưng
không cam kết giống từng bit vì trạng thái Dropout/TensorFlow giữa các tiến trình.


In [ ]:
if RUN_TRAINING:
    run_stage('train')
    state = json.loads((RUN / 'training_state.json').read_text())
    print('Step đã hoàn thành:', state['step'], '| Hoàn thành:', state['complete'])
    if not state['complete']:
        raise RuntimeError('Đã dừng ở giới hạn thời gian và lưu ZIP. Tiếp tục checkpoint trước khi chạy test.')
else:
    print('Chưa chạy train vì RUN_TRAINING=False.')


In [ ]:
# 14. QUÁ TRÌNH HỌC — CHỈ VALIDATION
if (RUN / 'training_history.csv').exists():
    history = pd.read_csv(RUN / 'training_history.csv')
    display(history.tail())
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for metric in ['OA', 'AA', 'Macro_F1', 'GM']:
        if metric in history: ax[0].plot(history.step, history[metric], label=metric)
    ax[0].set_title('Validation' if CFG['protocol'] == 'dip_train_only_validation' else 'Không có validation ở nhánh tái lập')
    if CFG['protocol'] == 'dip_train_only_validation': ax[0].legend()
    for loss in ['C_loss', 'D_loss', 'G_vs_C_loss', 'G_vs_D_loss']:
        ax[1].plot(history.step, history[loss], label=loss)
    ax[1].set_title('Loss — không tự suy ra chất lượng ảnh'); ax[1].legend()
    for a in ax: a.set_xlabel('Số lượt cập nhật classifier'); a.grid(alpha=.2)
    plt.tight_layout(); plt.savefig(RUN / 'validation_curves.png', dpi=140); plt.show()
else:
    print('Chưa có lịch sử huấn luyện.')

if (RUN / 'online_generated_counts.json').exists():
    generated = json.loads((RUN / 'online_generated_counts.json').read_text())
    display(pd.DataFrame({'class_id': generated['class_ids'], 'real_train': generated['real_train_counts'],
                          'fake_presentations_to_C': generated['generated_presentations']}).sort_values('class_id'))
    print('Đây là số lượt mẫu giả C đã học; không phải số ảnh giả độc nhất trong một bộ dữ liệu lưu sẵn.')


## 11. Mẫu sinh và chẩn đoán

Tạo 16 ví dụ mỗi lớp có đầu sinh; lớp đông nhất không cần đầu sinh. Dùng đúng checkpoint
đã chọn; so với mẫu train gần nhất, không lấy mẫu test để lọc. Hình thật/giả dùng chung
thang màu. Phổ vẽ từ phép nghịch đảo PCA là **phổ tái dựng**, không phải đo mới từ cảm biến.

Entropy trọng số, số mẫu hỗ trợ hiệu dụng, khoảng cách gần nhất và tỷ lệ phương sai chỉ
là dấu hiệu chẩn đoán; không phải thước đo chất lượng được hiệu chuẩn, chưa đủ kết luận
mode collapse hay ích lợi phân lớp. Không áp dụng FID/IS của ảnh RGB trực tiếp cho khối HSI.


In [ ]:
if RUN_TRAINING and json.loads((RUN / 'training_state.json').read_text())['complete']:
    run_stage('samples')
if (RUN / 'generation_diagnostics.csv').exists():
    display(pd.read_csv(RUN / 'generation_diagnostics.csv'))
    examples = np.load(RUN / 'generated_examples.npz')
    cid = int(examples['original_class_ids'][0])
    selection = np.flatnonzero(examples['original_class_ids'] == cid)[:3]
    real, fake = examples['nearest_real'][selection], examples['fake'][selection]
    component = 0
    lo, hi = min(real[..., component, 0].min(), fake[..., component, 0].min()), max(real[..., component, 0].max(), fake[..., component, 0].max())
    fig, ax = plt.subplots(2, 3, figsize=(10, 6))
    for i in range(3):
        ax[0, i].imshow(real[i, ..., component, 0], vmin=lo, vmax=hi, cmap='viridis')
        ax[0, i].set_title('Mẫu train gần nhất')
        ax[1, i].imshow(fake[i, ..., component, 0], vmin=lo, vmax=hi, cmap='viridis')
        ax[1, i].set_title('Mẫu GAN')
        ax[0, i].axis('off'); ax[1, i].axis('off')
    fig.suptitle(f'Lớp {cid}: {CLASS_NAMES[cid-1]} — thành phần PCA thứ nhất, chung thang màu')
    plt.tight_layout(); plt.savefig(RUN / 'generated_examples.png', dpi=140); plt.show()
    center = CFG['patch_size']//2
    real_spectrum = pca.inverse_transform(scaler.inverse_transform(real[:, center, center, :, 0]))
    fake_spectrum = pca.inverse_transform(scaler.inverse_transform(fake[:, center, center, :, 0]))
    plt.figure(figsize=(11, 4))
    plt.plot(real_spectrum[0], label='Mẫu train gần nhất — tái dựng PCA')
    plt.plot(fake_spectrum[0], label='Mẫu sinh — tái dựng PCA')
    plt.xlabel('Chỉ số dải phổ'); plt.ylabel('Giá trị tái dựng'); plt.legend(); plt.grid(alpha=.2)
    plt.title('Phổ tại tâm; không phải phép đo mới từ cảm biến')
    plt.tight_layout(); plt.savefig(RUN / 'generated_spectrum.png', dpi=140); plt.show()


## 12. Đánh giá cuối trên tập test đã giữ nguyên

OA: tỷ lệ đúng toàn bộ. AA: trung bình recall 16 lớp. Macro-F1: F1 trung bình các lớp.
GM: trung bình nhân recall (bằng 0 nếu có lớp recall bằng 0). Kappa: hệ số đồng thuận.
Luôn đọc số mẫu và recall từng lớp; lớp Grass-pasture-mowed chỉ có **2 mẫu test** nên
ước lượng rất nhạy. Không chọn lại checkpoint sau khi xem test.


In [ ]:
# 16. TEST CUỐI CÙNG — không chọn epoch, lượng sinh hoặc mô hình bằng kết quả này
if EVALUATE_TEST and RUN_TRAINING:
    run_stage('evaluate')
if (RUN / 'test_metrics.json').exists():
    result = json.loads((RUN / 'test_metrics.json').read_text())
    print('KẾT QUẢ DIP 16 LỚP — CÓ CHỒNG LẤN VÙNG ẢNH:')
    print(json.dumps(result, ensure_ascii=False, indent=2))
    display(pd.read_csv(RUN / 'test_per_class.csv'))
else:
    print('Chưa có kết quả test; không điền số minh họa.')


In [ ]:
# 17. BẢN ĐỒ CHỈ TRÊN TÂM TEST, KHÔNG TRỘN TRAIN VÀO ĐỘ CHÍNH XÁC
if (RUN / 'test_predictions.npz').exists():
    predictions = np.load(RUN / 'test_predictions.npz')
    q = predictions['coords']
    true_map = np.zeros_like(gt); predicted_map = np.zeros_like(gt); error_map = np.zeros_like(gt)
    true_map[q[:, 0], q[:, 1]] = predictions['true_class_ids']
    predicted_map[q[:, 0], q[:, 1]] = predictions['predicted_class_ids']
    error_map[q[:, 0], q[:, 1]] = np.where(predictions['true_class_ids'] == predictions['predicted_class_ids'], 1, 2)
    class_colors = [(0, 0, 0, 1)] + [plt.get_cmap('tab20')(i) for i in range(16)]
    cmap = ListedColormap(class_colors)
    fig, ax = plt.subplots(1, 4, figsize=(18, 5))
    ax[0].imshow(pseudo_rgb); ax[0].set_title('Ảnh giả màu')
    ax[1].imshow(true_map, cmap=cmap, vmin=0, vmax=16); ax[1].set_title('Nhãn thật — test')
    ax[2].imshow(predicted_map, cmap=cmap, vmin=0, vmax=16); ax[2].set_title('Dự đoán — test')
    ax[3].imshow(error_map, cmap=ListedColormap(['black', 'green', 'red']), vmin=0, vmax=2)
    ax[3].set_title('Xanh: đúng; đỏ: sai')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.savefig(RUN / 'test_maps.png', dpi=140); plt.show()
    cm = predictions['confusion_matrix']
    normalized = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(normalized, cmap='Blues', vmin=0, vmax=1)
    ids = predictions['probability_class_ids']
    ax.set_xticks(np.arange(len(ids)), ids); ax.set_yticks(np.arange(len(ids)), ids)
    ax.set_xlabel('Lớp dự đoán — ID gốc'); ax.set_ylabel('Lớp thật — ID gốc')
    for i in range(len(ids)):
        for j in range(len(ids)):
            ax.text(j, i, f'{normalized[i,j]:.2f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax); plt.tight_layout(); plt.savefig(RUN / 'test_confusion.png', dpi=140); plt.show()


## 13. Đối chiếu với bài báo — mốc tham khảo, không phải kiểm định hơn/kém

Bảng III trong [bản bài báo trên trang tác giả](https://profile.iiita.ac.in/srdubey/publications/2022_3D-HyperGAMO.pdf)
ghi Indian Pines: OA 85,95%, AA 82,18%, Kappa 0,8399.
Notebook in bên cạnh kết quả **thực sự chạy được**, không tự điền kết quả cũ.

Ngay cả khi dùng cùng mặt nạ, một seed/bước cuối trong môi trường mới không tương đương
kết quả tổng hợp của bài báo hoặc lần chọn điểm test cao nhất. Nhánh train-only/validation
còn thay đổi dữ liệu fit và lượng train. Các mốc chỉ giúp rà soát tái lập, không chứng minh
phương pháp mới vượt bài báo. Không trộn số của các bản thảo/bản xuất bản khác nhau.


In [ ]:
if (RUN / 'test_metrics.json').exists():
    result = json.loads((RUN / 'test_metrics.json').read_text())
    comparison = pd.DataFrame([
        {'Nguồn': 'Bài báo, Bảng III — tham khảo', 'OA (%)': 85.95, 'AA (%)': 82.18, 'Kappa': .8399},
        {'Nguồn': CFG['protocol'] + ', seed ' + str(CFG['train_seed']),
         'OA (%)': result['OA'] * 100, 'AA (%)': result['AA'] * 100, 'Kappa': result['Kappa']}
    ])
    display(comparison)
    comparison.to_csv(RUN / 'paper_reference_comparison.csv', index=False)
    print('Cách chọn checkpoint:', result['selection_rule'], '| Step:', result['selected_step'])
else:
    print('Chưa có test thực tế để đối chiếu.')


## 14. Ứng dụng minh họa trong notebook

Hai phần: (1) chọn lớp để xem mẫu đã sinh và C dự đoán lớp nào; (2) đưa khối HSI có
cùng 200 dải/cùng thứ tự và thang đo, chọn tọa độ để phân lớp bằng PCA/scaler đã lưu.
Đây là minh họa notebook, **chưa phải website hoàn chỉnh**.

3D-HyperGAMO sinh từ **nhiễu + nhãn + các mẫu train hỗ trợ**, không nhận một ảnh RGB
rồi biến ảnh đó thành ảnh mới. Điểm softmax không phải xác suất đúng đã hiệu chuẩn.
Mô hình không có lớp nền; không tự coi mọi pixel của một ảnh mới là một trong 16 vật liệu.


In [ ]:
def xem_mau_sinh(class_id=9, sample_index=0):
    path = RUN / 'generated_examples.npz'
    if not path.exists():
        print('Chưa có mẫu sinh. Hoàn thành bước 11 trước.'); return
    with np.load(path, allow_pickle=False) as e:
        indices = np.flatnonzero(e['original_class_ids'] == class_id)
        if not len(indices):
            print('Lớp này không có đầu sinh, hoặc không có trong ví dụ đã lưu.'); return
        i = indices[int(sample_index) % len(indices)]
        fake, real = e['fake'][i, ..., 0], e['nearest_real'][i, ..., 0]
        probability = e['probabilities'][i]
        pred = int(e['probability_class_ids'][probability.argmax()])
    fake_raw = pca.inverse_transform(scaler.inverse_transform(fake.reshape(-1, fake.shape[-1]))).reshape(11, 11, 200)
    real_raw = pca.inverse_transform(scaler.inverse_transform(real.reshape(-1, real.shape[-1]))).reshape(11, 11, 200)
    # Use one fixed display scaling derived from the source scene for both images.
    def rgb_from_reconstruction(cube):
        rgb = []
        for b in [30, 20, 10]:
            lo, hi = np.percentile(hsi[..., b], [2, 98])
            rgb.append(np.clip((cube[..., b] - lo) / (hi-lo+1e-8), 0, 1))
        return np.stack(rgb, -1)
    fig, ax = plt.subplots(1, 2, figsize=(7, 3))
    ax[0].imshow(rgb_from_reconstruction(real_raw)); ax[0].set_title('Train gần nhất — tái dựng')
    ax[1].imshow(rgb_from_reconstruction(fake_raw)); ax[1].set_title('Mẫu GAN — tái dựng')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()
    print('Lớp điều kiện:', class_id, CLASS_NAMES[class_id-1])
    print('C dự đoán:', pred, CLASS_NAMES[pred-1], '| Điểm softmax:', round(float(probability.max()), 4))
    print('Ví dụ đã sinh ở bước 11, không phải một phép đo chất lượng độc lập.')

def phan_lop_hsi(cube, coords):
    cube = np.asarray(cube)
    coords = np.asarray(coords)
    if cube.ndim != 3 or cube.shape[-1] != preprocessing['raw_bands'] or not np.isfinite(cube).all():
        raise ValueError('Cần khối HSI (H,W,200) hữu hạn, cùng thứ tự dải và thang đo.')
    if coords.ndim != 2 or coords.shape[1] != 2 or len(coords) == 0 or len(coords) > 2000:
        raise ValueError('Nhập 1–2000 tọa độ (hàng, cột) cho một lượt minh họa.')
    if not (RUN / 'test_complete.json').exists():
        raise RuntimeError('Hoàn thành đánh giá cố định trước khi minh họa ứng dụng.')
    transformed = scaler.transform(pca.transform(cube.reshape(-1, cube.shape[-1]))).reshape(*cube.shape[:2], -1)
    patches = extract_patches(transformed, coords, CFG['patch_size'])
    folder = RUN / 'application'; folder.mkdir(exist_ok=True)
    import uuid
    request_id = uuid.uuid4().hex[:12]
    input_path, output_path = folder / (request_id + '_input.npz'), folder / (request_id + '_predictions.npz')
    np.savez_compressed(input_path, patches=patches, coords=coords)
    run_stage('infer', ['--input', str(input_path), '--output', str(output_path)])
    with np.load(output_path, allow_pickle=False) as output:
        prob, ids = output['probabilities'], output['class_ids']
    return pd.DataFrame({'row': coords[:, 0], 'col': coords[:, 1],
                         'predicted_class_id': ids[prob.argmax(1)], 'softmax_score': prob.max(1)})

if (RUN / 'samples_complete.json').exists():
    xem_mau_sinh(class_id=9, sample_index=0)
# Tùy chọn sau khi train xong, không tự chạy thêm khi Run All:
# display(phan_lop_hsi(hsi, np.array([[65, 23], [100, 100]], dtype=int)))


## 15. Lưu kết quả và tiếp tục sau khi hết giờ

Chỉ tải **một ZIP** dưới đây: nguồn mã, cấu hình, phiên bản thư viện, audit nhãn/chồng lấn,
chỉ số chia tập, PCA/scaler, cache, checkpoint C/D/G/đầu sinh/optimizer, nhật ký, ảnh và
độ đo test. Để tiếp tục, gắn ZIP đó vào Input rồi điền `RESUME_ZIP` ở ô 1.
Giữ nguyên cấu hình và môi trường; chỉ được đổi giới hạn thời gian, GPU và cờ tiếp tục.
Checkpoint không biến một lần chạy chưa đủ bước thành kết quả hoàn thành.

ZIP tự cập nhật tại các mốc lưu, nhưng không thể tự cứu khi máy ảo bị tắt cưỡng bức.
Lưu Version có Output và tải file trước khi rời phiên; nếu liên kết tải báo 404, tải qua
bảng Output của phiên bản đã lưu. Không cần huấn luyện lại chỉ để xem file đã có.


In [ ]:
archive_path = zip_run()
print('TẢI FILE NÀY:', archive_path)
print('Dung lượng:', round(archive_path.stat().st_size / 1024**2, 1), 'MiB')
display(FileLink(str(archive_path)))
print('Hoàn thành huấn luyện:', (RUN / 'training_state.json').exists() and json.loads((RUN / 'training_state.json').read_text())['complete'])
print('Có test cuối:', (RUN / 'test_metrics.json').exists())


## Giới hạn và bước nghiên cứu tiếp theo

- Giữ đủ 16 lớp, bao gồm các lớp rất hiếm; một vài lớp có quá ít test để kết luận ổn định.
- DIP ở đây không trùng tâm train/test nhưng có chồng lấn vùng ảnh. Nhánh mặc định còn fit
  tiền xử lý toàn ảnh. Không gọi điểm của nhánh này là kết quả tổng quát hóa không gian sạch.
- Không thể bảo đảm 16 lớp đều có train/test 11×11 không dùng chung pixel: một số lớp hiếm
  chỉ nằm trong vùng nhỏ. Bản 8 lớp có khoảng đệm là thí nghiệm phụ với phạm vi khác.
- Một seed là lần chạy nền, chưa đủ chứng minh độ ổn định. Chưa có so sánh C-only/ROS cùng
  giao thức nên chưa quy mọi thay đổi độ chính xác cho GAN.
- Sau baseline ổn, giữ nguyên giao thức để làm đối chứng và thử **một cải tiến nhỏ có căn cứ**.
  Kiểm soát cách đánh giá và cải tiến mô hình là hai việc khác nhau, không tự đổi tên đề tài.
